1. Инициализация и загрузка данных
2. Очистка и предподготовка корпуса НТД
3. Инструменты поиска НТД
4. Классификатор значимости абзацев ПД
5. Baseline (базовый режим)
6. Сравнение методов поиска кандидатов НТД
7. Прямое формирование контекста из корпуса ПД
8. Нормативно-ориентированное формирование контекста
9. Финальный эксперимент: сравнение трёх режимов

In [4]:
import os
import shutil
from google.colab import drive
from google.colab import userdata

mountpoint = '/content/drive'

try:
    drive.flush_and_unmount()
    print('Drive unmounted')
except Exception as e:
    print('Unmount info:', e)

if os.path.exists(mountpoint):
    try:
        shutil.rmtree(mountpoint)
        print('Mountpoint removed')
    except Exception as e:
        print('Cannot remove mountpoint:', e)

os.makedirs(mountpoint, exist_ok=True)
print('Mountpoint recreated, contents:', os.listdir(mountpoint))

drive.mount(mountpoint, force_remount=True)

Drive unmounted
Mountpoint recreated, contents: []
Mounted at /content/drive


# 1. ИНИЦИАЛИЗАЦИЯ ОКРУЖЕНИЯ И ЗАГРУЗКА ДАННЫХ

In [ ]:
import os, shutil, json, pickle, re, time, math
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import requests
from google.colab import drive, userdata
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from scipy.stats import loguniform
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
import joblib

!pip install -q pymorphy3 rank_bm25
import pymorphy3
morph = pymorphy3.MorphAnalyzer()
try:
    from rank_bm25 import BM25Okapi
    BM25_AVAILABLE = True
except:
    BM25_AVAILABLE = False

#  КОНФИГУРАЦИЯ
BASE_DIR = "/content/drive/MyDrive/ВКР_Егорова"
PKL_PD = os.path.join(BASE_DIR, "ПД/эмбеддинги/1. ОБУСТРОЙСТВО КОВЫКТИНСКОГО.pkl")
NTD_DIR = os.path.join(BASE_DIR, "СП/ntd_embeded")
RESULTS_DIR = os.path.join(BASE_DIR, "Результаты/Итог")
EMBEDDINGS_DIR = os.path.join(BASE_DIR, "ПД/эмбеддинги")
os.makedirs(RESULTS_DIR, exist_ok=True)

YANDEX_MODEL = "yandexgpt-lite"
TOP_K = 3
LLM_MAX_TOKENS = 2000
TEMPERATURE = 0.1

# Монтирование Google Диска
mountpoint = '/content/drive'
try:
    drive.flush_and_unmount()
except:
    pass
if os.path.exists(mountpoint):
    try:
        shutil.rmtree(mountpoint)
    except:
        pass
os.makedirs(mountpoint, exist_ok=True)
drive.mount(mountpoint, force_remount=True)

# ЗАГРУЗКА ЭМБЕДДИНГОВ
def load_pkl_embeddings(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    ids = [str(item["id"]) for item in data]
    texts = [item["text"] for item in data]
    X = np.array([item["embedding"] for item in data], dtype=np.float32)
    return ids, texts, X

def load_ntd_dir_strict(ntd_dir, expected_dim):
    if not os.path.exists(ntd_dir):
        raise FileNotFoundError(f"НТД не найдена: {ntd_dir}")
    all_ids, all_texts, all_vecs = [], [], []
    for fn in sorted(os.listdir(ntd_dir)):
        if not fn.lower().endswith(".pkl"):
            continue
        with open(os.path.join(ntd_dir, fn), "rb") as f:
            data = pickle.load(f)
        tag = os.path.splitext(fn)[0]
        for idx, item in enumerate(data):
            if not isinstance(item, dict):
                continue
            raw_id = str(item.get("id", f"idx_{idx}"))
            txt = item.get("text", "")
            emb = item.get("embedding", None)
            if emb is None:
                continue
            v = np.array(emb, dtype=np.float32).ravel()
            if v.size != expected_dim or not np.isfinite(v).all():
                continue
            all_ids.append(f"{tag}::{raw_id}")
            all_texts.append(txt)
            all_vecs.append(v)
    X = np.vstack(all_vecs).astype(np.float32)
    return all_ids, all_texts, X

def l2_normalize(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    return mat / (norms + 1e-12)

print("Загрузка данных...")
ids_pd, texts_pd, X_pd = load_pkl_embeddings(PKL_PD)
ids_ntd, texts_ntd, X_ntd = load_ntd_dir_strict(NTD_DIR, expected_dim=X_pd.shape[1])
X_pd = l2_normalize(X_pd)
X_ntd = l2_normalize(X_ntd)
print(f"Загружено: ПД={len(texts_pd)} абзацев, НТД={len(texts_ntd)} фрагментов")

# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
def call_yandexgpt(system, user, max_tokens=500):
    api_key = userdata.get('yandex_key')
    folder_id = userdata.get('YANDEX_FOLDER_ID')
    url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Api-Key {api_key}",
        "x-folder-id": folder_id,
    }
    messages = []
    if system:
        messages.append({"role": "system", "text": system})
    messages.append({"role": "user", "text": user})
    data = {
        "modelUri": f"gpt://{folder_id}/{YANDEX_MODEL}",
        "completionOptions": {"stream": False, "temperature": TEMPERATURE, "maxTokens": max_tokens},
        "messages": messages
    }
    try:
        resp = requests.post(url, headers=headers, json=data, timeout=90)
        if resp.status_code != 200:
            return f"API_ERROR: {resp.status_code}"
        return resp.json()["result"]["alternatives"][0]["message"]["text"]
    except Exception as e:
        return f"API_ERROR: {e}"

def is_header_like(text):
    t = text.strip()
    if not t:
        return False
    if len(t) < 80 and t.endswith(":"):
        return True
    if re.fullmatch(r"^[A-ZА-Я0-9\.\- ]{3,50}$", t):
        return True
    return False

def build_struct_blocks(texts):
    n = len(texts)
    starts = [0]
    for i, t in enumerate(texts):
        if i == 0:
            continue
        if is_header_like(t):
            starts.append(i)
    starts = sorted(set(starts))
    blocks = []
    for i, s in enumerate(starts):
        e = starts[i+1] - 1 if (i+1) < len(starts) else (n - 1)
        blocks.append((s, e))
    return blocks

def find_block_for_index(blocks, idx):
    for (s, e) in blocks:
        if s <= idx <= e:
            return (s, e)
    return (0, len(texts_pd) - 1)

def tokenize_russian(text):
    text = text.lower()
    text = re.sub(r"[^\w\s\-]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def safe_truncate(s, max_len):
    return s if len(s) <= max_len else (s[:max_len] + "…")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 29.8 MB/s eta 0:00:00
Mounted at /content/drive
Загрузка данных...
Загружено: ПД=565 абзацев, НТД=111289 фрагментов


# 2. ОЧИСТКА И ПРЕДПОДГОТОВКА КОРПУСА НТД

In [ ]:
# 1. Предварительное извлечение текста из HTML-фрагментов
def extract_text_from_html(text):
    """Если строка содержит HTML-теги, пытаемся извлечь из неё чистый текст."""
    t = str(text).strip()
    if not (t.startswith('<table') or t.startswith('<tr') or t.startswith('<td')):
        return t
    try:
        from bs4 import BeautifulSoup
        soup = BeautifulSoup(t, 'html.parser')
        plain = soup.get_text(separator=' ', strip=True)
        if len(plain) >= 10:
            return plain
        else:
            return ''
    except ImportError:
        return t

texts_ntd = [extract_text_from_html(t) for t in texts_ntd]

# 2. Основной фильтр содержательных фрагментов
def is_relevant_ntd(text):
    t = str(text).strip()
    low = t.lower()

    if len(t) < 30:
        modal_words = ['должен', 'должна', 'должно', 'должны',
                       'следует', 'не допускается', 'запрещается',
                       'необходимо', 'обязательно', 'разрешается']
        if any(m in low for m in modal_words):
            return True
        return False

    trash = [
        "комментарий к статье", "утратил силу", "см. предыдущую редакцию",
        "принят государственной думой", "в редакции, введенной в действие",
        "вступает в силу", "см. текст статьи", "см. комментарии"
    ]
    if any(x in low for x in trash):
        if "вступает в силу" in low and len(t) > 100:
            modal_words = ['должен', 'должна', 'должно', 'должны',
                           'следует', 'не допускается', 'запрещается']
            if any(m in low for m in modal_words):
                return True
        return False

    if re.fullmatch(r"(статья|глава|раздел)\s+[\d\.]+\s*", low):
        return False

    if re.match(r'^см\.\s', low):
        return False

    return True


# 3. Применение фильтра
keep = [i for i, t in enumerate(texts_ntd) if is_relevant_ntd(t)]
print(f"Осталось {len(keep)} из {len(texts_ntd)} фрагментов НТД")

clean_to_orig = {i: orig for i, orig in enumerate(keep)}
orig_to_clean = {orig: i for i, orig in enumerate(keep)}

texts_ntd_clean = [texts_ntd[i] for i in keep]
ids_ntd_clean = [ids_ntd[i] for i in keep]
X_ntd_clean = X_ntd[keep]


# 4. Токенизация и BM25
print("Предподготовка BM25-индекса...")
def preprocess_fast(text):
    words = re.findall(r'[а-яё]+', text.lower())
    return [morph.parse(w)[0].normal_form for w in words if len(w) > 1]

clean_tokens = [preprocess_fast(t) for t in texts_ntd_clean]

df_counts = defaultdict(int)
for doc in clean_tokens:
    for term in set(doc):
        df_counts[term] += 1
N = len(clean_tokens)
idf = {term: math.log((N - freq + 0.5) / (freq + 0.5) + 1) for term, freq in df_counts.items()}

inverted = defaultdict(list)
for i, doc in enumerate(clean_tokens):
    for term in set(doc):
        inverted[term].append(i)

doc_len = np.array([len(d) for d in clean_tokens])
avgdl = doc_len.mean()

def compute_bm25_scores_for_text(text, k1=1.5, b_param=0.75):
    query_tokens = preprocess_fast(text)
    if not query_tokens:
        return np.zeros(N)
    scores = np.zeros(N)
    for term in query_tokens:
        if term not in idf:
            continue
        for i in inverted.get(term, []):
            tf = clean_tokens[i].count(term)
            scores[i] += idf[term] * (tf * (k1 + 1)) / (tf + k1 * (1 - b_param + b_param * doc_len[i] / avgdl))
    return scores


# 5. Маппинг пунктов и типов документов
clause_to_indices = defaultdict(list)
for orig_idx, nid in enumerate(ids_ntd):
    if '::' in nid:
        base = nid.rsplit('::', 1)[0]
        clause_to_indices[base].append(orig_idx)
    else:
        clause_to_indices[nid].append(orig_idx)

type_to_clean_fixed = defaultdict(list)
for clean_i, orig_i in enumerate(keep):
    nid = ids_ntd[orig_i]
    nid_upper = nid.upper()
    if 'ГОСТ' in nid_upper: dtype = 'ГОСТ'
    elif 'СНИП' in nid_upper: dtype = 'СНиП'
    elif 'САНПИН' in nid_upper: dtype = 'СанПиН'
    elif 'СТО_' in nid_upper or 'СТО ' in nid_upper: dtype = 'СТО'
    elif 'СТУ' in nid_upper: dtype = 'СТУ'
    elif nid_upper.startswith('СП') or 'СП ' in nid_upper or 'СП_' in nid_upper: dtype = 'СП'
    elif nid_upper.endswith('-ФЗ'): dtype = 'ФЗ'
    else: dtype = 'ПРОЧЕЕ'
    type_to_clean_fixed[dtype].append(clean_i)

print("Предподготовка завершена.")


Очистка корпуса НТД...
Осталось 100993 из 111289 фрагментов НТД
Предподготовка BM25-индекса...
Предподготовка завершена.


# 3. ИНСТРУМЕНТЫ ПОИСКА НТД

In [ ]:
# Тематические профили
THEMATIC_PROFILES = {
    'пожарная безопасность': {
        'keywords': ['пожарный', 'пожарной', 'пожарная', 'огнетушитель', 'гидрант',
                     'противопожарный', 'эвакуация', 'огнестойкость', 'возгорание',
                     'тупиковый проезд', 'разворотная площадка', 'пожарное депо',
                     'пожарных', 'пожарные', 'пожарного', 'тупиковый', 'разворотный',
                     'проезд', '15х15', '15x15', 'взрывопожароопасн', 'категория помещения'],
        'docs': ['СП 1.13130', 'СП 2.13130', 'СП 4.13130', 'СП 7.13130', 'СП 8.13130',
                 'СП 9.13130', 'СП 10.13130', 'СП 12.13130', 'СП 155.13130',
                 'СП 231.1311500', 'СП 485.1311500', '123-ФЗ', 'Постановление №1479']
    },
    'генплан и транспорт': {
        'keywords': ['генплан', 'дорога', 'въезд', 'земельный участок', 'благоустройство',
                     'вертикальная планировка', 'площадка', 'тротуар', 'обочина', 'кювет',
                     'уклон', 'автодорога', 'бортовой камень', 'радиус поворота'],
        'docs': ['СП 18.13330', 'СП 34.13330', 'СП 37.13330', 'СП 42.13330',
                 'СНиП II-89-80', 'СНиП 2.05.07-91', 'Постановление №160']
    },
    'геология, грунты и фундаменты': {
        'keywords': ['грунт', 'фундамент', 'свая', 'мерзлота', 'промерзание', 'основание',
                     'оттаивание', 'сейсмичность', 'геология', 'гидрогеология', 'котлован',
                     'насыпь', 'засыпка', 'вечномерзлый'],
        'docs': ['СП 20.13330', 'СП 22.13330', 'СП 24.13330', 'СП 25.13330',
                 'СП 26.13330', 'СП 45.13330', 'СП 47.13330', 'СП 50-101',
                 'СП 11-105-97']
    },
    'строительные конструкции и материалы': {
        'keywords': ['конструкция', 'сталь', 'металлоконструкции', 'бетон', 'железобетон',
                     'коррозия', 'антикоррозийная', 'нагрузка', 'пролет', 'ферма', 'колонна',
                     'ограждение', 'барьерн', 'асфальт', 'покрытие'],
        'docs': ['СП 14.13330', 'СП 16.13330', 'СП 28.13330', 'СП 53-102',
                 'СП 56.13330', 'СП 78.13330', 'СНиП 2.03.11-85']
    },
    'теплоизоляция и климатология': {
        'keywords': ['теплоизоляция', 'тепловой изоляции', 'толщина изоляции', 'теплопровод',
                     'минераловат', 'пенополистирол', 'плотность теплового потока', 'климат',
                     'температура воздуха', 'ветер', 'снеговой район'],
        'docs': ['СП 50.13330', 'СП 61.13330', 'СП 124.13330', 'СП 131.13330',
                 'СНиП 23-02-2003']
    },
    'водоснабжение и канализация': {
        'keywords': ['канализация', 'сточные воды', 'канализационные', 'очистные сооружения',
                     'напорная канализация', 'дождевые стоки', 'КОС', 'дренаж', 'водоснабжение',
                     'водопровод', 'питьевая', 'водозабор', 'насосная', 'РЧВ', 'гидрант'],
        'docs': ['СП 30.13330', 'СП 31.13330', 'СП 32.13330', 'СП 40.13330',
                 'СНиП 2.04.02-84', 'СанПиН 2.1.4.1110-02', 'Постановление №219', '74-ФЗ']
    },
    'отопление и вентиляция': {
        'keywords': ['отопление', 'вентиляция', 'кондиционирование', 'воздухообмен',
                     'приточная', 'вытяжная', 'теплопотери', 'микроклимат', 'калорифер',
                     'кратность обмена', 'аспирация', 'дымоудаление', 'теплосеть'],
        'docs': ['СП 60.13330', 'СП 73.13330', 'СНиП 3.05.03-85']
    },
    'промышленная безопасность и газоснабжение': {
        'keywords': ['газопровод', 'нефтепровод', 'магистральный', 'ГРС', 'КС', 'давление газа',
                     'охранная зона', 'диаметр трубы', 'категория участка', 'газораспределение',
                     'опасный производственный объект', 'ОПО', 'технологический трубопровод'],
        'docs': ['СП 35.13330', 'СП 36.13330', 'СП 284.1325800', '116-ФЗ',
                 'СП 231.1311500', 'Приказ Ростехнадзора № 533', 'Приказ Ростехнадзора № 534',
                 'Приказ Ростехнадзора № 536']
    },
    'экология и охрана окружающей среды': {
        'keywords': ['экология', 'окружающая среда', 'ПДВ', 'СЗЗ', 'санитарно-защитная',
                     'отходы', 'выбросы', 'загрязнение', 'рекультивация', 'ООС', 'ПМООС',
                     'рыбохозяйственный', 'водный объект'],
        'docs': ['7-ФЗ', '52-ФЗ', '73-ФЗ', '89-ФЗ', '96-ФЗ',
                 'СанПиН 2.2.1/2.1.1.1200-03', 'СанПиН 1.2.3685-21',
                 'Постановление №913', 'Постановление №800',
                 'Приказом Госкомрыболовства № 154', 'Постановление №1614']
    },
    'стандарты пао газпром': {
        'keywords': ['газпром', 'сто газпром', 'дочернее общество', 'норма технологического проектирования',
                     'нтп', 'технологический регламент', 'коррозия', 'магистральный транспорт',
                     'компрессорная станция', 'укпг', 'скважина', 'ковякин'],
        'docs': ['СТО СМК 17-2008', 'СТО Газпром РД 1.14-139-2005', 'СТО Газпром НТП 1.8-001-2004',
                 'СТО Газпром 9.2-003-2020', 'СТО Газпром 9.2-002-2019', 'СТО Газпром 9.2-002-2009',
                 'СТО Газпром 9.1-035-2014', 'СТО Газпром 9.0-001-2009', 'СТО Газпром 2-4.1-971-2015',
                 'СТО Газпром 2-4.1-212-2008', 'СТО Газпром 2-3.5-454-2010', 'СТО Газпром 2-3.5-354-2009',
                 'СТО Газпром 2-2.4-083-2006', 'СТО Газпром 2-2.2-136-2007', 'СТО Газпром 2-2.1-607-2011',
                 'СТО Газпром 2-1.12-434-2010', 'СТО Газпром 12-1.1-026-2020', 'СТО Газпром 11-2005',
                 'СТО Газпром 097-2011', 'СТО Газпром 089-2010']
    },
    'федеральные законы и общие нормы': {
        'keywords': ['федеральный закон', 'технический регламент', 'законодательство',
                     'нормативно-правовой акт', 'ФЗ'],
        'docs': ['96-ФЗ', '89-ФЗ', '74-ФЗ', '73-ФЗ', '7-ФЗ', '52-ФЗ', '116-ФЗ',
                 '384-ФЗ', '190-ФЗ', '184-ФЗ', '155-ФЗ', '151-ФЗ', '123-ФЗ',
                 '117-ФЗ', '212-ФЗ', '200-ФЗ', '191-ФЗ', '187-ФЗ', '33-ФЗ',
                 '4979-ФЗ', '412-ФЗ']
    }
}

def detect_theme(text):
    text_lower = text.lower()
    theme_scores = {}
    for theme, profile in THEMATIC_PROFILES.items():
        score = sum(1 for kw in profile['keywords'] if kw.lower() in text_lower)
        if score > 0:
            theme_scores[theme] = score
    return sorted(theme_scores.items(), key=lambda x: -x[1])

# Работа с документами
def normalize_doc_name(name):
    """Приводит название документа к единому нормализованному виду:
    убирает пробелы, дефисы, подчёркивания и точки, переводит в нижний регистр."""
    return re.sub(r'[\s_\-\.]+', '', name).lower()

def find_document_indices(doc_ref):
    doc_ref_norm = normalize_doc_name(doc_ref)
    m = re.search(r'(сп|гост|снип|санпин|сто|сту)\s*([\d\.]+)', doc_ref_norm)
    indices = []
    for i, nid in enumerate(ids_ntd):
        nid_norm = normalize_doc_name(nid)
        if m:
            typ, num = m.group(1), m.group(2)
            if typ in nid_norm and re.search(rf'(^|[^0-9]){re.escape(num)}([^0-9]|$)', nid_norm):
                indices.append(i)
        else:
            if doc_ref_norm in nid_norm:
                indices.append(i)
    return indices

def extract_refs_with_clauses(text):
    results = []
    pattern1 = r'(СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+.*?(?:п\.|пункт\s*)\s*(\d+(?:\.\d+)*)'
    for doc_type, clause_num in re.findall(pattern1, text, re.IGNORECASE):
        doc_match = re.search(rf'({doc_type}\s*[\d\.\-]+)', text, re.IGNORECASE)
        if doc_match:
            results.append((doc_match.group(1).strip(), clause_num))
    pattern2 = r'(?:п\.|пункт\s*)\s*(\d+(?:\.\d+)*).*?(ГОСТ|СП|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+'
    for clause_num, doc_type in re.findall(pattern2, text, re.IGNORECASE):
        doc_match = re.search(rf'({doc_type}\s*[\d\.\-]+)', text, re.IGNORECASE)
        if doc_match:
            doc_ref = doc_match.group(1).strip()
            if not any(d == doc_ref and c == clause_num for d, c in results):
                results.append((doc_ref, clause_num))
    return results

def find_clause_in_ntd_flexible(doc_ref, clause_num):
    doc_ref_clean = re.sub(r'\s+', ' ', doc_ref).strip().lower()
    parts = clause_num.split('.')
    patterns = [
        rf'\b{re.escape(clause_num)}\b', rf'\b{re.escape(clause_num)}\.\s',
        rf'\({re.escape(clause_num)}\)', rf'^{re.escape(clause_num)}\s',
        rf'^{re.escape(clause_num)}\.\s', rf'п\.?\s*{re.escape(clause_num)}',
        rf'пункт\s*{re.escape(clause_num)}',
    ]
    if len(parts) >= 1:
        patterns.append(rf'^{parts[0]}\.\s')
        patterns.append(rf'^{parts[0]}\.\d')
    found = []
    for i, nid in enumerate(ids_ntd):
        if doc_ref_clean not in re.sub(r'\s+', ' ', nid).lower():
            continue
        text_lower = texts_ntd[i].lower()
        for pattern in patterns:
            if re.search(pattern, text_lower):
                found.append(i)
                break
    if not found and len(parts) >= 2:
        parent = '.'.join(parts[:-1])
        for i, nid in enumerate(ids_ntd):
            if doc_ref_clean not in re.sub(r'\s+', ' ', nid).lower():
                continue
            text_lower = texts_ntd[i].lower()
            if re.search(rf'\b{re.escape(parent)}\b', text_lower) or \
               re.search(rf'^{re.escape(parent)}\s', text_lower) or \
               re.search(rf'^{re.escape(parent)}\.\s', text_lower):
                found.append(i)
    return found

def fuzzy_find_clause_v2(doc_ref, clause_num, max_distance=0.5):
    doc_indices = find_document_indices(doc_ref)
    if not doc_indices:
        return []
    clause_candidates = defaultdict(list)
    for idx in doc_indices:
        text = texts_ntd[idx]
        for cl in re.findall(r'(?:п\.?\s*)?(\d+(?:\.\d+)+)\b', text):
            clause_candidates[cl].append(idx)
        for cl in re.findall(r'^(\d+(?:\.\d+)*)\s', text, re.MULTILINE):
            if '.' in cl:
                clause_candidates[cl].append(idx)
    if not clause_candidates:
        return []
    clause_parts = clause_num.split('.')
    best_match, best_score = None, 0
    for cl in clause_candidates:
        score = 0
        if cl == clause_num: score = 100
        elif clause_num in cl or cl in clause_num: score = 90
        elif cl.startswith(clause_parts[0] + '.'): score = 80
        else: score = SequenceMatcher(None, clause_num, cl).ratio() * 50
        if score > best_score:
            best_score, best_match = score, cl
    if best_score >= max_distance * 100 and best_match:
        return clause_candidates[best_match]
    return []

def extract_technical_parameters(text):
    params = []
    patterns = [
        (r'(разворотн\w*\s*площа\w*)', r'(\d+[хx×]\d+\s*(?:метр|м)\w*)'),
        (r'(проез\w*)', r'(ширин\w*\s*(?:не\s*менее|не\s*более)?\s*\d+[.,]?\d*\s*(?:метр|м)\w*)'),
        (r'(расстоян\w*)', r'(?:не\s*менее|не\s*более|принято)\s*(\d+[.,]?\d*\s*(?:метр|м)\w*)'),
        (r'(высот\w*\s*огражден\w*)', r'(\d+[.,]?\d*\s*(?:метр|м)\w*)'),
        (r'(глубин\w*\s*(?:промерзан|оттаиван)\w*)', r'(\d+[.,]?\d*\s*(?:метр|м)\w*)'),
        (r'(площад\w*)', r'(?:более|менее|не\s*менее)\s*(\d+[.,]?\d*\s*(?:га|м2|м²)\w*)'),
    ]
    for ep, vp in patterns:
        em = re.search(ep, text, re.IGNORECASE)
        vm = re.search(vp, text, re.IGNORECASE)
        if em and vm:
            params.append((em.group(1), vm.group(1)))
    return params

# Модально-числовые бусты
UNIT_TUPLE_RE = re.compile(r"(\d+[.,]?\d*)\s*(мм|см|м|км|мпа|м2|м3|%|кВт|В|А|°с|°C|т|кг|л|сек|мин|час|сут|м/с|г/л|мг/л)", re.IGNORECASE)
UNIT_RE = re.compile(r"\d+[.,]?\d*\s*(?:м|мм|см|км|мпа|м2|м3|%|кВт|В|А|°с|°C|т|кг|л|сек|мин|час|сут|м/с|г/л|мг/л)", re.IGNORECASE)

def compute_modality_numerical_boost(ntd_text, pd_text):
    boost = 1.0
    modal_words = re.findall(r'(должен|должны|следует|требуется|необходимо|не допускается|запрещается)', ntd_text, re.IGNORECASE)
    if modal_words:
        boost *= 1.2
    pd_numbers = set(UNIT_TUPLE_RE.findall(pd_text))
    ntd_numbers = set(UNIT_TUPLE_RE.findall(ntd_text))
    if pd_numbers:
        for pd_val, pd_unit in pd_numbers:
            try: pd_val_float = float(pd_val.replace(',', '.'))
            except: continue
            for nt_val, nt_unit in ntd_numbers:
                if pd_unit.lower() == nt_unit.lower():
                    try:
                        nt_val_float = float(nt_val.replace(',', '.'))
                        if abs(pd_val_float - nt_val_float) / max(abs(nt_val_float), 1e-6) < 0.1:
                            boost *= 1.5
                            break
                    except: pass
    return boost

def section_boost(nid):
    path = nid.lower()
    keys = ['генплан','планиров','транспорт','проезд','подъезд','дорог','въезд','выезд','пожар']
    return 1.7 if any(k in path for k in keys) else 1.0

def get_document_priorities(pd_text):
    priority_docs = []
    text_lower = pd_text.lower()
    routing_rules = [
        (['проезд', 'подъезд', 'пожарн', 'ширина', 'дорог', 'въезд', 'выезд'], ['СП 42.13330', 'СП 4.13130', 'СП 34.13330']),
        (['генплан', 'площадка', 'размещение', 'зонирование'], ['СП 18.13330', 'СП 42.13330']),
        (['трубопровод', 'газопровод', 'нефтепровод'], ['СП 36.13330', 'СП 34.13330']),
        (['фундамент', 'свай', 'грунт', 'геолог'], ['СП 22.13330', 'СП 24.13330', 'СП 47.13330']),
        (['пожар', 'противопожарный', 'огнетушитель', 'гидрант', 'эвакуация'], ['СП 1.13130', 'СП 2.13130', 'СП 4.13130', 'СП 8.13130', 'СП 155.13130']),
        (['канализация', 'сточные воды', 'КОС', 'очистные'], ['СП 32.13330', 'СП 30.13330']),
        (['тепло', 'отопление', 'вентиляция', 'кондиционирование'], ['СП 60.13330', 'СП 50.13330']),
    ]
    for keywords, docs in routing_rules:
        if any(kw in text_lower for kw in keywords):
            priority_docs.extend(docs)
    return list(set(priority_docs))

# Расширенный поиск (multi-query + RM3)
def generate_multi_query(text):
    queries = [text]
    params = extract_technical_parameters(text)
    imperatives = re.findall(r'(должен|должны|следует|требуется|необходимо|обеспечить|не допускается)', text, re.IGNORECASE)
    tech_terms = [kw for kw in THEMATIC_PROFILES['пожарная безопасность']['keywords'] +
                  THEMATIC_PROFILES['генплан и транспорт']['keywords'] if kw in text.lower()]
    if params:
        for entity, value in params:
            queries.append(f"требования к {entity} {value}")
            queries.append(f"минимальное {entity} {value}")
    if imperatives:
        queries.append(f"нормативные требования {' '.join(set(imperatives))}")
    if tech_terms:
        queries.append(f"технические нормы для {' '.join(tech_terms[:5])}")
    synonym_map = {
        'въезд': ['выезд', 'примыкание', 'подъезд', 'въезд-выезд'],
        'проезд': ['подъезд', 'проезжая часть', 'дорога', 'полоса движения'],
        'пожарный': ['противопожарный', 'огнеборцы'],
        'ширина': ['габарит', 'ширина проезжей части'],
        'площадка': ['территория', 'участок', 'земельный участок'],
    }
    for key, synonyms in synonym_map.items():
        if key in text.lower():
            queries.append(f"{key} {' '.join(synonyms[:2])}")
    return list(dict.fromkeys(queries))[:5]

def rrf_combine_scores(results_lists, k=60):
    rrf_scores = defaultdict(float)
    for lst in results_lists:
        for rank, item in enumerate(lst):
            doc_idx = item[0] if isinstance(item, tuple) else item
            rrf_scores[doc_idx] += 1.0 / (k + rank + 1)
    return rrf_scores

def extract_informative_terms(texts_list, top_n=15):
    from collections import Counter
    term_counter = Counter()
    modal_boost = re.compile(r'(должен|следует|не допускается|не менее|не более)', re.IGNORECASE)
    for text in texts_list:
        tokens = preprocess_fast(text)
        for tok in tokens:
            term_counter[tok] += 1
        for i in range(len(tokens)-1):
            term_counter[f"{tokens[i]}_{tokens[i+1]}"] += 1
        if modal_boost.search(text):
            for tok in tokens:
                term_counter[tok] += 2
    most_common = [term for term, _ in term_counter.most_common(top_n*2)]
    stop_words = {'и', 'в', 'на', 'с', 'по', 'для', 'не', 'что', 'как', 'к', 'от', 'из', 'при', 'без', 'до', 'под', 'над', 'за', 'или', 'а', 'то', 'это'}
    return ' '.join([term for term in most_common if term not in stop_words and len(term) > 2][:top_n])

def search_improved(pd_idx, top_k=50):
    text = texts_pd[pd_idx]
    refs = re.findall(r"(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+", text, re.IGNORECASE)
    if refs:
        return search_universal_enhanced(pd_idx, top_k)

    queries = generate_multi_query(text)
    results_per_query = []
    for q in queries:
        scores = compute_bm25_scores_for_text(q)
        top_idx = np.argsort(scores)[-50:][::-1]
        results_per_query.append([clean_to_orig[i] for i in top_idx if scores[i] > 0])
    rrf_scores = rrf_combine_scores(results_per_query)
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: -x[1])
    top_docs = [doc for doc, _ in sorted_docs[:20]]

    top_doc_texts = [texts_ntd[doc] for doc in top_docs[:10]]
    expanded_query = text + ' ' + extract_informative_terms(top_doc_texts, top_n=10)
    scores_rm3 = compute_bm25_scores_for_text(expanded_query)

    priority_docs = get_document_priorities(text)
    for i in range(len(scores_rm3)):
        nid = ids_ntd[clean_to_orig[i]] if i < len(clean_to_orig) else ""
        for pattern in priority_docs:
            if pattern.lower() in nid.lower():
                scores_rm3[i] *= 3.0

    themes = detect_theme(text)
    if themes:
        main_theme = themes[0][0]
        theme_docs = THEMATIC_PROFILES[main_theme]['docs']
        for i in range(len(scores_rm3)):
            nid = ids_ntd[clean_to_orig[i]] if i < len(clean_to_orig) else ""
            for doc_pattern in theme_docs:
                if doc_pattern.lower() in nid.lower():
                    scores_rm3[i] *= 2.0
                    break

    top_indices = np.argsort(scores_rm3)[-50:][::-1]
    return [clean_to_orig[i] for i in top_indices if scores_rm3[i] > 0]

#  HyDE
def generate_hypothetical_requirement(text):
    prompt = f"""Ты эксперт по нормативно-технической документации в нефтегазовой отрасли.
Проанализируй абзац проектной документации и представь, какое нормативное требование (из СП, ГОСТ, СНиП и т.п.)
могло бы проверяться на соответствие данному решению.
Сформулируй это требование в виде одного-двух предложений, используя характерный для нормативных документов язык
(с указанием параметров, допусков, расстояний, если они упомянуты).

Абзац ПД:
"{text[:1500]}"

Гипотетическое требование:"""
    try:
        resp = call_yandexgpt("", prompt, max_tokens=200)
        if not resp.startswith("API_ERROR"):
            return resp.strip()
    except:
        pass
    return ""

def hyde_search(pd_idx, top_k=50):
    text = texts_pd[pd_idx]
    refs = re.findall(r"(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+", text, re.IGNORECASE)
    if refs:
        return search_universal_enhanced(pd_idx, top_k)
    hypothetical = generate_hypothetical_requirement(text)
    if not hypothetical:
        return search_universal_enhanced(pd_idx, top_k)

    scores_bm25 = compute_bm25_scores_for_text(hypothetical)
    cos_scores = np.dot(X_ntd_clean, X_pd[pd_idx])

    def normalize(v):
        v_min, v_max = v.min(), v.max()
        return (v - v_min) / (v_max - v_min + 1e-8)
    combined = 0.65 * normalize(cos_scores) + 0.35 * normalize(scores_bm25)

    themes = detect_theme(text)
    if themes:
        main_theme = themes[0][0]
        theme_docs = THEMATIC_PROFILES[main_theme]['docs']
        for i in range(len(combined)):
            nid = ids_ntd_clean[i] if i < len(ids_ntd_clean) else ""
            for doc_pattern in theme_docs:
                if doc_pattern.lower() in nid.lower():
                    combined[i] *= 2.0
                    break

    top_doc_indices = np.argsort(combined)[-50:][::-1]
    return [keep[i] for i in top_doc_indices if combined[i] > 0]

#  Regex-канал
def extract_constraints_slots(text):
    """Извлекает слоты с атрибутом, оператором, числом и единицей.
    Сохраняет исходное строковое представление числа (val_raw) для поиска с запятой."""
    slots = []
    pattern = r"(ширин\w*|высот\w*|расстоян\w*|глубин\w*|площад\w*|радиус\w*|ширин\w*\s*проезд\w*|проез\w*|дорог\w*|въезд\w*|выезд\w*)" \
              r"\s*(не\s*менее|не\s*более|более|менее|не\s*менее|не\s*более|принято|принята|принят|приняты|до|от)\s*" \
              r"(\d+[.,]?\d*)\s*(мм|см|м|км|мпа|м2|м3|га|%|кВт|В|А|°с|°C|т|кг|л|сек|мин|час|сут)"
    for match in re.finditer(pattern, text, re.IGNORECASE):
        attr, op, val_str, unit = match.groups()
        val_raw = val_str
        try:
            val_float = float(val_str.replace(',', '.'))
        except:
            continue
        op = op.strip().lower()
        if 'не менее' in op or op.endswith('не менее'): op = '>='
        elif 'не более' in op or op.endswith('не более'): op = '<='
        elif op == 'более' or op.endswith('более'): op = '>'
        elif op == 'менее' or op.endswith('менее'): op = '<'
        elif 'до' in op: op = '<='
        elif 'от' in op: op = '>='
        else: op = '='
        slots.append({'attr': attr.lower(), 'op': op, 'val': val_float, 'val_raw': val_raw, 'unit': unit.lower()})
    return slots

def extract_domain_keywords(text):
    keys = set()
    t = text.lower()
    if re.search(r'въезд|выезд|примыкание|подъезд', t):
        keys.update(['въезд', 'выезд', 'примыкание', 'подъезд'])
    if re.search(r'проезд|проезжая часть|дорог|полоса движения', t):
        keys.update(['проезд', 'проезжая часть', 'дорог', 'полоса движения'])
    if re.search(r'пожарн|огнетушитель|гидрант|противопожарный', t):
        keys.update(['пожарн', 'пожарный', 'противопожарный', 'гидрант'])
    if re.search(r'ширина|габарит', t):
        keys.update(['ширина', 'габарит'])
    if re.search(r'площадка|территория|участок', t):
        keys.update(['площадка', 'территория', 'участок'])
    words = re.findall(r'\b[а-яё]{4,}\b', t)
    for w in words:
        if w not in ['который','которые','также','следует','должен','должны','этого','всего']:
            keys.add(w)
    return list(keys)[:15]

def heuristic_score(ntd_text, keys, slots):
    score = 0
    tl = ntd_text.lower()
    for k in keys:
        if k in tl:
            score += 1
    units = set(s['unit'] for s in slots)
    for u in units:
        if u in tl:
            score += 2
    for s in slots:
        if s['unit'] in tl:
            found = False
            if re.search(rf'{re.escape(s["val_raw"])}\s*{re.escape(s["unit"])}', tl):
                found = True
            if not found and ',' in s['val_raw']:
                alt_val = s['val_raw'].replace(',', '.')
                if re.search(rf'{re.escape(alt_val)}\s*{re.escape(s["unit"])}', tl):
                    found = True
            if not found and '.' in s['val_raw']:
                alt_val = s['val_raw'].replace('.', ',')
                if re.search(rf'{re.escape(alt_val)}\s*{re.escape(s["unit"])}', tl):
                    found = True
            if found and s['attr'] in tl:
                score += 3
    return score

def regex_candidates(pd_idx, top_k=50):
    text = texts_pd[pd_idx]
    slots = extract_constraints_slots(text)
    keys = extract_domain_keywords(text)
    if not slots and not keys:
        return []
    hits = []
    for i, t in enumerate(texts_ntd_clean):
        s = heuristic_score(t, keys, slots)
        if s > 0:
            hits.append((keep[i], s))
    hits.sort(key=lambda x: -x[1])
    return [h[0] for h in hits[:top_k]]

#  Универсальный улучшенный поиск (используется при явных ссылках)
def search_universal_enhanced(pd_idx, top_k=50):
    text = texts_pd[pd_idx]
    refs = re.findall(r"(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+", text, re.IGNORECASE)
    refs_with_clauses = extract_refs_with_clauses(text)
    target_doc = None
    target_clause = None
    if refs_with_clauses:
        target_doc = refs_with_clauses[0][0]
        target_clause = refs_with_clauses[0][1]
    elif refs:
        target_doc = refs[0]

    candidates = set()
    if target_doc:
        doc_indices = find_document_indices(target_doc)
        if doc_indices:
            if target_clause:
                exact = find_clause_in_ntd_flexible(target_doc, target_clause)
                if exact: candidates.update(exact)
                else:
                    fuzzy = fuzzy_find_clause_v2(target_doc, target_clause)
                    if fuzzy: candidates.update(fuzzy)
                    else: candidates.update(find_similar_in_document_v2(target_doc, text, pd_idx, top_k=15))
            else:
                candidates.update(find_similar_in_document_v2(target_doc, text, pd_idx, top_k=15))

    if not candidates:
        tokens = preprocess_fast(text)
        bigrams = [f"{tokens[i]}_{tokens[i+1]}" for i in range(len(tokens)-1)]
        expanded_query = " ".join(tokens + bigrams)
        scores = compute_bm25_scores_for_text(expanded_query)
        themes = detect_theme(text)
        if themes:
            main_theme = themes[0][0]
            theme_docs = THEMATIC_PROFILES[main_theme]['docs']
            for i in range(len(scores)):
                nid = ids_ntd[clean_to_orig[i]] if i < len(clean_to_orig) else ""
                for doc_pattern in theme_docs:
                    if doc_pattern.lower() in nid.lower():
                        scores[i] *= 3.0
                        break
        for dtype in ['СП', 'ГОСТ', 'СНиП', 'СанПиН', 'СТО', 'СТУ']:
            for i in type_to_clean_fixed.get(dtype, []):
                scores[i] *= 1.5
        top = np.argsort(scores)[-20:][::-1]
        doc_candidates = [clean_to_orig[i] for i in top if scores[i] > 0]
        for doc_idx in doc_candidates:
            nid = ids_ntd[doc_idx]
            base = nid.rsplit('::', 1)[0] if '::' in nid else nid
            clause_indices = clause_to_indices.get(base, [doc_idx])
            if len(clause_indices) > 1:
                clause_vecs = X_ntd[clause_indices]
                sims = np.dot(clause_vecs, X_pd[pd_idx])
                top3 = np.argsort(sims)[-3:][::-1]
                for i in top3: candidates.add(clause_indices[i])
            else: candidates.add(doc_idx)
    if not candidates:
        return []
    expanded = set()
    for idx in candidates:
        nid = ids_ntd[idx]
        base = nid.rsplit('::', 1)[0] if '::' in nid else nid
        expanded.update(clause_to_indices.get(base, [idx]))
    expanded_list = list(expanded)
    final_scores = [float(np.dot(X_ntd[i], X_pd[pd_idx])) for i in expanded_list]
    top = np.argsort(final_scores)[-top_k:][::-1]
    return [expanded_list[i] for i in top]

def find_similar_in_document_v2(doc_ref, target_text, pd_idx, top_k=10):
    doc_indices = find_document_indices(doc_ref)
    if not doc_indices: return []
    if len(doc_indices) <= top_k: return doc_indices
    keywords = []
    nums = re.findall(r'\d+[.,]?\d*\s*(?:га|м|метр|км|см|мм)', target_text, re.IGNORECASE)
    keywords.extend(nums)
    words = re.findall(r'[а-яё]{4,}', target_text.lower())
    stop_words = {'который', 'которые', 'также', 'следует', 'должен', 'должны', 'этого', 'всего'}
    keywords.extend([w for w in words if w not in stop_words][:10])
    scores = np.zeros(len(doc_indices))
    for i, idx in enumerate(doc_indices):
        text_lower = texts_ntd[idx].lower()
        for kw in keywords:
            if kw.lower() in text_lower: scores[i] += 1
        for phrase in nums:
            if phrase.lower() in text_lower: scores[i] += 3
    if scores.max() == 0:
        for i, idx in enumerate(doc_indices):
            scores[i] = float(np.dot(X_ntd[idx], X_pd[pd_idx]))
    top_local = np.argsort(scores)[-top_k:][::-1]
    result = [doc_indices[i] for i in top_local if scores[i] > 0]
    return result if result else doc_indices[:top_k]

# Финальный поиск (RRF + бусты)
def get_ntd_candidates_final(pd_idx, top_k=8):
    res1 = search_improved(pd_idx, top_k=50)
    res2 = hyde_search(pd_idx, top_k=50)
    res3 = regex_candidates(pd_idx, top_k=50)
    lists_for_rrf = [res1, res2, res3]
    lists_for_rrf = [lst for lst in lists_for_rrf if lst]
    if not lists_for_rrf:
        return []
    rrf_scores = rrf_combine_scores(lists_for_rrf, k=60)
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: -x[1])
    top_docs = [doc for doc, _ in sorted_docs[:60]]

    final_scores = []
    text = texts_pd[pd_idx]
    priority_docs = get_document_priorities(text)
    slots = extract_constraints_slots(text)

    for doc_idx in top_docs:
        cos_sim = float(np.dot(X_ntd[doc_idx], X_pd[pd_idx]))
        mod_num_boost = compute_modality_numerical_boost(texts_ntd[doc_idx], text)
        sec_boost = section_boost(ids_ntd[doc_idx])
        priority_boost = 1.5 if any(p.lower() in ids_ntd[doc_idx].lower() for p in priority_docs) else 1.0
        constraint_boost = 1.0
        if slots:
            ntd_text = texts_ntd[doc_idx].lower()
            for s in slots:
                if s['unit'] not in ntd_text:
                    continue
                found_number = False
                if re.search(rf'{re.escape(s["val_raw"])}\s*{re.escape(s["unit"])}', ntd_text):
                    found_number = True
                if not found_number and ',' in s['val_raw']:
                    alt_val = s['val_raw'].replace(',', '.')
                    if re.search(rf'{re.escape(alt_val)}\s*{re.escape(s["unit"])}', ntd_text):
                        found_number = True
                if not found_number and '.' in s['val_raw']:
                    alt_val = s['val_raw'].replace('.', ',')
                    if re.search(rf'{re.escape(alt_val)}\s*{re.escape(s["unit"])}', ntd_text):
                        found_number = True

                if found_number:
                    if s['attr'] in ntd_text:
                        constraint_boost = 1.6
                        break

        total = cos_sim * mod_num_boost * sec_boost * priority_boost * constraint_boost
        final_scores.append((doc_idx, total))

    final_scores.sort(key=lambda x: -x[1])
    seen_bases = set()
    unique_candidates = []
    for doc_idx, score in final_scores:
        nid = ids_ntd[doc_idx]
        base = nid.rsplit('::', 1)[0] if '::' in nid else nid
        if base not in seen_bases:
            seen_bases.add(base)
            unique_candidates.append(doc_idx)
        if len(unique_candidates) >= top_k:
            break
    return [{"id": ids_ntd[idx], "text": texts_ntd[idx], "sim": float(np.dot(X_ntd[idx], X_pd[pd_idx]))} for idx in unique_candidates]

# 4. КЛАССИФИКАТОР ЗНАЧИМОСТИ АБЗАЦЕВ ПД

In [ ]:
def build_features_for_texts(texts):
    n = len(texts)
    df_all = pd.DataFrame({'text': texts})
    df_all['text_len'] = df_all['text'].apply(lambda t: len(str(t)) if t is not None else 0)
    df_all['norm_refs'] = df_all['text'].apply(lambda t: count_occurrences(t, r'(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+'))
    df_all['imperative'] = df_all['text'].apply(lambda t: count_occurrences(t, r'(должен|должны|следует|требуется|необходимо|обеспечить|не допускается)'))
    df_all['unit_count'] = df_all['text'].apply(unit_count_func)
    df_all['num_count'] = df_all['text'].apply(lambda t: len(re.findall(r'\d', str(t))) if t is not None else 0)
    df_all['num_density'] = df_all['num_count'] / (df_all['text_len'] + 1)
    df_all['is_header'] = df_all['text'].apply(is_header_safe)
    df_all['has_clause'] = df_all['text'].apply(has_clause_func)
    df_all['tech_terms'] = df_all['text'].apply(count_tech_terms_safe)
    df_all['position'] = np.arange(n) / max(1, n-1)
    df_all['after_header'] = df_all['is_header'].shift(1, fill_value=0)
    return df_all[feature_columns]

def count_occurrences(text, pattern):
    try:
        if not isinstance(text, str):
            return 0
        return len(re.findall(pattern, text, re.IGNORECASE))
    except:
        return 0

def unit_count_func(text):
    try:
        if not isinstance(text, str):
            return 0
        return len(UNIT_RE.findall(text))
    except:
        return 0

def is_header_safe(text):
    try:
        if text is None or not isinstance(text, str):
            return 0
        t = text.strip()
        if not t:
            return 0
        return int(len(t) < 80 and (t.endswith(':') or re.fullmatch(r"^[A-ZА-Я0-9\.\- ]{3,50}$", t)))
    except:
        return 0

def has_clause_func(text):
    try:
        if not isinstance(text, str):
            return 0
        return int(bool(re.search(r'(?:п\.|пункт)\s*\d+', text, re.IGNORECASE)))
    except:
        return 0

In [ ]:
# 1. Загрузка Ковыкты
LABELS_KOVYKTA = os.path.join(BASE_DIR, "ПД/итоговые метки_со значимостью.xlsx")
df_kov = pd.read_excel(LABELS_KOVYKTA)
df_kov['embedding_np'] = list(X_pd)
df_kov.rename(columns={'значимый / незначимый': 'label'}, inplace=True)
df_kov['text'] = df_kov['text'].apply(lambda x: str(x) if pd.notna(x) else '')
y_kov = df_kov['label'].astype(int).values

# 2. Загрузка Чаянды
PKL_CHAYANDA = os.path.join(BASE_DIR, "ПД/эмбеддинги/2. ОБУСТРОЙСТВО ЧАЯНДИНСКОГО НГКМ.pkl")
LABELS_CHAYANDA = os.path.join(BASE_DIR, "ПД/chayandinskoe_id_text_for_manual_labeling.xlsx")

ids_ch, texts_ch, X_ch = load_pkl_embeddings(PKL_CHAYANDA)
X_ch = l2_normalize(X_ch)

df_ch = pd.read_excel(LABELS_CHAYANDA)
df_ch.rename(columns={'значимый / незначимый': 'label'}, inplace=True)
df_ch['text'] = df_ch['text'].apply(lambda x: str(x) if pd.notna(x) else '')
y_ch = df_ch['label'].astype(int).values

# 3. Признаки (базовые + новые)
feature_columns = [
    'norm_refs', 'imperative', 'unit_count', 'text_len', 'num_count', 'num_density',
    'is_header', 'has_clause', 'tech_terms', 'position', 'after_header'
]

features_kov = build_features_for_texts(texts_pd)
features_ch = build_features_for_texts(texts_ch)

def add_extra_text_features(texts, feature_df):
    modal_pattern = r'\b(должен|должна|должно|должны|следует|не допускается|запрещается|необходимо|обязательно)\b'
    ntd_ref_pattern = r'(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+'
    unit_number_pattern = r'\d+[\.,]?\d*\s*(?:м|мм|см|км|кг|г|т|с|мин|ч|°С|%|Вт|кВт|МПа|кПа|Па|л|м³|кг/м³)'
    feature_df = feature_df.copy()
    feature_df['num_modal'] = [len(re.findall(modal_pattern, t.lower())) for t in texts]
    feature_df['num_ntd_refs'] = [len(re.findall(ntd_ref_pattern, t, re.IGNORECASE)) for t in texts]
    feature_df['has_unit_number'] = [1 if re.search(unit_number_pattern, t) else 0 for t in texts]
    return feature_df

features_kov = add_extra_text_features(texts_pd, features_kov)
features_ch = add_extra_text_features(texts_ch, features_ch)
feature_columns = list(features_kov.columns)


blocks_kov = build_struct_blocks(texts_pd)
group_ids = np.zeros(len(texts_pd), dtype=int)
for bi, (s, e) in enumerate(blocks_kov):
    group_ids[s:e+1] = bi

found_split = False
RANDOM_STATE_KOV = 42
gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=RANDOM_STATE_KOV)
train_kov_idx, test_kov_idx = next(gss.split(X_pd, y_kov, groups=group_ids))

if not found_split:
    print("ВНИМАНИЕ: в тестовой выборке Ковыкты может не быть одного из классов")
    train_kov_idx, test_kov_idx = next(gss.split(X_pd, y_kov, groups=group_ids))

# Валидация из train Ковыкты (стратифицированно)
sss_val = StratifiedShuffleSplit(n_splits=1, test_size=0.15/0.70, random_state=42)
train_kov_sub, val_kov_sub = next(sss_val.split(np.zeros(len(train_kov_idx)), y_kov[train_kov_idx]))
train_kov_idx_final = train_kov_idx[train_kov_sub]
val_kov_idx = train_kov_idx[val_kov_sub]


# 5. Разбиение Чаянды (70/30) + создание смешанной валидации
sss_ch = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
train_ch_idx, test_ch_idx = next(sss_ch.split(X_ch, y_ch))

# Чтобы валидация была более репрезентативной, добавим часть Чаянды в val
val_ch_idx, _ = next(sss_ch.split(X_ch[train_ch_idx], y_ch[train_ch_idx]))
val_ch_idx_global = train_ch_idx[val_ch_idx]


# 6. Масштабирование признаков на объединённом train
X_train_features = np.vstack([features_kov.iloc[train_kov_idx_final].values,
                              features_ch.iloc[train_ch_idx].values])
scaler_extra = StandardScaler()
scaler_extra.fit(X_train_features.astype(np.float32))

X_extra_kov = scaler_extra.transform(features_kov[feature_columns].astype(np.float32))
X_extra_ch = scaler_extra.transform(features_ch[feature_columns].astype(np.float32))

# Базовые расширенные матрицы (эмбеддинги + признаки)
X_enhanced_kov = np.hstack([X_pd, X_extra_kov])
X_enhanced_ch = np.hstack([X_ch, X_extra_ch])

X_train = np.vstack([X_enhanced_kov[train_kov_idx_final], X_enhanced_ch[train_ch_idx]])
y_train = np.concatenate([y_kov[train_kov_idx_final], y_ch[train_ch_idx]])

X_val = np.vstack([X_enhanced_kov[val_kov_idx], X_enhanced_ch[val_ch_idx_global]])
y_val = np.concatenate([y_kov[val_kov_idx], y_ch[val_ch_idx_global]])

X_test_kov = X_enhanced_kov[test_kov_idx]
y_test_kov = y_kov[test_kov_idx]
X_test_ch = X_enhanced_ch[test_ch_idx]
y_test_ch = y_ch[test_ch_idx]

print(f"Train: {X_train.shape[0]} (Ковыкта {len(train_kov_idx_final)} + Чаянда {len(train_ch_idx)})")
print(f"Val:   {X_val.shape[0]}")
print(f"Test Ковыкта: {X_test_kov.shape[0]}, Test Чаянда: {X_test_ch.shape[0]}")
print(f"Распределение классов: train 0/1 = {np.bincount(y_train)}")
print(f"Распределение классов: val   0/1 = {np.bincount(y_val)}")
print(f"Распределение классов: test_kov 0/1 = {np.bincount(y_test_kov)}")
print(f"Распределение классов: test_ch  0/1 = {np.bincount(y_test_ch)}")

# Модель 1: LightGBM
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'scale_pos_weight': scale_pos,
    'lambda_l1': 0.1,
    'lambda_l2': 1.0,
    'random_state': 42,
    'verbosity': -1,
    'early_stopping_rounds': 50
}
lgb_train = lgb.Dataset(X_train, y_train)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
lgb_model = lgb.train(params, lgb_train, valid_sets=[lgb_val], valid_names=['val'])

probs_val_lgb = lgb_model.predict(X_val)
best_f1_lgb, best_thr_lgb = -1, 0.5
for thr in np.linspace(0.1, 0.9, 81):
    preds = (probs_val_lgb >= thr).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1_lgb:
        best_f1_lgb, best_thr_lgb = f1, thr
print(f"LightGBM: порог={best_thr_lgb:.2f}, F1 val={best_f1_lgb:.3f}")

# Модель 2: Базовый MLP
base_mlp = MLPClassifier(hidden_layer_sizes=(256,128), alpha=0.0001, learning_rate_init=0.0005,
                         activation='relu', solver='adam', max_iter=300, early_stopping=True,
                         validation_fraction=0.1, random_state=42)
base_mlp.fit(X_train, y_train)
probs_val_base = base_mlp.predict_proba(X_val)[:,1]
best_f1_base, best_thr_base = -1, 0.5
for thr in np.linspace(0.1,0.9,81):
    preds = (probs_val_base >= thr).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1_base:
        best_f1_base, best_thr_base = f1, thr
print(f"Базовый MLP: порог={best_thr_base:.2f}, F1 val={best_f1_base:.3f}")


# Модель 3: Улучшенный MLP (PCA + TF‑IDF + SMOTE)
train_texts_kov = [texts_pd[i] for i in train_kov_idx_final]
train_texts_ch = [texts_ch[i] for i in train_ch_idx]
train_texts_all = train_texts_kov + train_texts_ch
train_emb_all = np.vstack([X_pd[train_kov_idx_final], X_ch[train_ch_idx]])

tfidf = TfidfVectorizer(max_features=100, tokenizer=tokenize_russian)
tfidf.fit(train_texts_all)
tfidf_kov = tfidf.transform(texts_pd).toarray()
tfidf_ch = tfidf.transform(texts_ch).toarray()

pca = PCA(n_components=0.95, random_state=42)
pca.fit(train_emb_all)
X_emb_pca_kov = pca.transform(X_pd)
X_emb_pca_ch = pca.transform(X_ch)

X_enhanced_kov_v2 = np.hstack([X_emb_pca_kov, X_extra_kov, tfidf_kov])
X_enhanced_ch_v2 = np.hstack([X_emb_pca_ch, X_extra_ch, tfidf_ch])

X_train_v2 = np.vstack([X_enhanced_kov_v2[train_kov_idx_final], X_enhanced_ch_v2[train_ch_idx]])
y_train_v2 = np.concatenate([y_kov[train_kov_idx_final], y_ch[train_ch_idx]])

X_val_v2 = np.vstack([X_enhanced_kov_v2[val_kov_idx], X_enhanced_ch_v2[val_ch_idx_global]])
y_val_v2 = np.concatenate([y_kov[val_kov_idx], y_ch[val_ch_idx_global]])

X_test_kov_v2 = X_enhanced_kov_v2[test_kov_idx]
y_test_kov_v2 = y_kov[test_kov_idx]
X_test_ch_v2 = X_enhanced_ch_v2[test_ch_idx]
y_test_ch_v2 = y_ch[test_ch_idx]

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_v2, y_train_v2)

param_dist = {
    'hidden_layer_sizes': [(256,128), (512,256)],
    'alpha': loguniform(1e-4, 1e-2),
    'learning_rate_init': loguniform(1e-4, 1e-2),
}
mlp_improved = MLPClassifier(activation='relu', solver='adam', max_iter=500,
                             early_stopping=True, validation_fraction=0.1, random_state=42)
random_search = RandomizedSearchCV(mlp_improved, param_distributions=param_dist,
                                   n_iter=10, cv=3, scoring='f1', random_state=42, n_jobs=-1, verbose=1)
random_search.fit(X_train_res, y_train_res)
best_mlp_imp = random_search.best_estimator_
print(f"Улучшенный MLP: {random_search.best_params_}, CV F1={random_search.best_score_:.3f}")

probs_val_imp = best_mlp_imp.predict_proba(X_val_v2)[:,1]
best_f1_imp, best_thr_imp = -1, 0.5
for thr in np.linspace(0.1,0.9,81):
    preds = (probs_val_imp >= thr).astype(int)
    f1 = f1_score(y_val_v2, preds, zero_division=0)
    if f1 > best_f1_imp:
        best_f1_imp, best_thr_imp = f1, thr
print(f"Улучшенный MLP: порог={best_thr_imp:.2f}, F1 val={best_f1_imp:.3f}")


# Ансамбль (LightGBM + Базовый MLP) – ручное усреднение
def ensemble_predict_proba(X):
    p_lgb = lgb_model.predict(X)
    p_mlp = base_mlp.predict_proba(X)[:, 1]
    return (p_lgb + p_mlp) / 2.0

probs_val_ens = ensemble_predict_proba(X_val)
best_f1_ens, best_thr_ens = -1, 0.5
for thr in np.linspace(0.1, 0.9, 81):
    preds = (probs_val_ens >= thr).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best_f1_ens:
        best_f1_ens, best_thr_ens = f1, thr
print(f"Ансамбль (LGB+MLP): порог={best_thr_ens:.2f}, F1 val={best_f1_ens:.3f}")


results = []
test_configs = [
    ("LightGBM", lgb_model, X_test_kov, y_test_kov, best_thr_lgb, "Ковыкта", "predict"),
    ("LightGBM", lgb_model, X_test_ch, y_test_ch, best_thr_lgb, "Чаянда", "predict"),
    ("Базовый MLP", base_mlp, X_test_kov, y_test_kov, best_thr_base, "Ковыкта", "proba"),
    ("Базовый MLP", base_mlp, X_test_ch, y_test_ch, best_thr_base, "Чаянда", "proba"),
    ("Улучшенный MLP", best_mlp_imp, X_test_kov_v2, y_test_kov_v2, best_thr_imp, "Ковыкта", "proba"),
    ("Улучшенный MLP", best_mlp_imp, X_test_ch_v2, y_test_ch_v2, best_thr_imp, "Чаянда", "proba"),
    ("Ансамбль", None, X_test_kov, y_test_kov, best_thr_ens, "Ковыкта", "ensemble"),
    ("Ансамбль", None, X_test_ch, y_test_ch, best_thr_ens, "Чаянда", "ensemble"),
]

for name, model, X_t, y_t, thr, test_name, mode in test_configs:
    if mode == "proba":
        probs = model.predict_proba(X_t)[:, 1]
    elif mode == "predict":
        probs = model.predict(X_t)
    elif mode == "ensemble":
        probs = ensemble_predict_proba(X_t)
    else:
        continue
    preds = (probs >= thr).astype(int)
    results.append({"Модель": name, "Тест": test_name,
                    "Precision": precision_score(y_t, preds, zero_division=0),
                    "Recall": recall_score(y_t, preds, zero_division=0),
                    "F1": f1_score(y_t, preds, zero_division=0),
                    "ROC-AUC": roc_auc_score(y_t, probs)})
df_res = pd.DataFrame(results)
print("\n=== ИТОГОВОЕ СРАВНЕНИЕ НА ТЕСТЕ ===")
print(df_res.to_string(index=False))


# Финальный классификатор (Ансамбль)
print(f"\nФинальная модель: Ансамбль LGB+MLP (порог={best_thr_ens:.2f})")
X_all_kov = X_enhanced_kov
all_probs = ensemble_predict_proba(X_all_kov)

candidate_indices = [i for i, p in enumerate(all_probs) if p >= best_thr_ens]
sig_indices = candidate_indices
print(f"Проверяемых абзацев (без пост-фильтра): {len(sig_indices)}")


Обучение классификатора значимости на смешанных данных...
ВНИМАНИЕ: в тестовой выборке Ковыкты может не быть одного из классов
Train: 732 (Ковыкта 305 + Чаянда 427)
Val:   382
Test Ковыкта: 176, Test Чаянда: 184
Распределение классов: train 0/1 = [374 358]
Распределение классов: val   0/1 = [215 167]
Распределение классов: test_kov 0/1 = [93 83]
Распределение классов: test_ch  0/1 = [113  71]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


LightGBM: порог=0.30, F1 val=0.934
Базовый MLP: порог=0.35, F1 val=0.862


/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Улучшенный MLP: {'alpha': np.float64(0.00020511104188433984), 'hidden_layer_sizes': (256, 128), 'learning_rate_init': np.float64(0.0008288916866885145)}, CV F1=0.799
Улучшенный MLP: порог=0.46, F1 val=0.871
Ансамбль (LGB+MLP): порог=0.50, F1 val=0.933

=== ИТОГОВОЕ СРАВНЕНИЕ НА ТЕСТЕ ===
        Модель    Тест  Precision   Recall       F1  ROC-AUC
      LightGBM Ковыкта   0.000000 0.000000 0.000000 0.711362
      LightGBM  Чаянда   0.670000 0.943662 0.783626 0.894553
   Базовый MLP Ковыкта   0.893333 0.807229 0.848101 0.919161
   Базовый MLP  Чаянда   0.692308 0.887324 0.777778 0.887573
Улучшенный MLP Ковыкта   0.965517 0.674699 0.794326 0.940148
Улучшенный MLP  Чаянда   0.730769 0.802817 0.765101 0.906269
      Ансамбль Ковыкта   1.000000 0.216867 0.356436 0.919679
      Ансамбль  Чаянда   0.737500 0.830986 0.781457 0.897794

Финальная модель: Ансамбль LGB+MLP (порог=0.50)
Проверяемых абзацев (без пост-фильтра): 277


In [ ]:
TECH_KEYWORDS = [
    "грунт","глубин","свай","бетон","фундамент","отметк","уклон","сталь","кабель",
    "трубопровод","арматур","клапан","давлен","температур","мощность","расстояни",
    "пожарн","защит","сигнализаци","объект","сооружени","здани"
]
def count_tech_terms(text):
    try:
        t = text.lower()
        return sum(1 for kw in TECH_KEYWORDS if kw in t)
    except: return 0

In [ ]:
# ФИНАЛЬНАЯ МОДЕЛЬ: БАЗОВЫЙ MLP
print("\n" + "="*60)
print("ФИНАЛЬНАЯ МОДЕЛЬ: БАЗОВЫЙ MLP")
print("="*60)

# Применяем базовый MLP ко всей Ковыкте
X_all_kov = X_enhanced_kov
probs_all = base_mlp.predict_proba(X_all_kov)[:, 1]
preds_all = (probs_all >= best_thr_base).astype(int)

print(f"Значимых абзацев (базовый MLP, без пост‑фильтра): {preds_all.sum()} из {len(preds_all)}")

sig_indices = [i for i, p in enumerate(probs_all) if p >= best_thr_base]
print(f"Итоговый sig_indices содержит {len(sig_indices)} абзацев")


SAVE_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(base_mlp, os.path.join(SAVE_DIR, "base_mlp.pkl"))
joblib.dump(scaler_extra, os.path.join(SAVE_DIR, "scaler_extra.pkl"))

with open(os.path.join(SAVE_DIR, "inference_params.json"), "w", encoding="utf-8") as f:
    json.dump({
        "best_threshold": best_thr_base,
        "feature_columns": feature_columns
    }, f, ensure_ascii=False, indent=2)

print(f"Модель (базовый MLP) и параметры сохранены в {SAVE_DIR}")


ФИНАЛЬНАЯ МОДЕЛЬ: БАЗОВЫЙ MLP
Значимых абзацев (базовый MLP, без пост‑фильтра): 379 из 565
Итоговый sig_indices содержит 379 абзацев
Модель (базовый MLP) и параметры сохранены в /content/drive/MyDrive/ВКР_Егорова/models


In [ ]:
# 1. Загрузка модели и параметров

BASE_DIR = "/content/drive/MyDrive/ВКР_Егорова"
SAVE_DIR = os.path.join(BASE_DIR, "models")

base_mlp = joblib.load(os.path.join(SAVE_DIR, "base_mlp.pkl"))
scaler_extra = joblib.load(os.path.join(SAVE_DIR, "scaler_extra.pkl"))

with open(os.path.join(SAVE_DIR, "inference_params.json"), "r", encoding="utf-8") as f:
    params = json.load(f)

best_thr = params["best_threshold"]
feature_columns = params["feature_columns"]

# 2. Загрузка эмбеддингов и текстов Ковыкты
PKL_PD = os.path.join(BASE_DIR, "ПД/эмбеддинги/1. ОБУСТРОЙСТВО КОВЫКТИНСКОГО.pkl")

def load_pkl_embeddings(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    ids = [str(item["id"]) for item in data]
    texts = [item["text"] for item in data]
    X = np.array([item["embedding"] for item in data], dtype=np.float32)
    return ids, texts, X

ids_pd, texts_pd, X_pd = load_pkl_embeddings(PKL_PD)
X_pd = X_pd / (np.linalg.norm(X_pd, axis=1, keepdims=True) + 1e-12)


# 3. Вспомогательные функции
TECH_KEYWORDS = [
    "грунт","глубин","свай","бетон","фундамент","отметк","уклон","сталь","кабель",
    "трубопровод","арматур","клапан","давлен","температур","мощность","расстояни",
    "пожарн","защит","сигнализаци","объект","сооружени","здани"
]

def count_tech_terms_safe(text):
    try:
        t = text.lower()
        return sum(1 for kw in TECH_KEYWORDS if kw in t)
    except:
        return 0

def count_occurrences(text, pattern):
    try:
        if not isinstance(text, str):
            return 0
        return len(re.findall(pattern, text, re.IGNORECASE))
    except:
        return 0

UNIT_RE = re.compile(r"\d+[.,]?\d*\s*(?:м|мм|см|км|мпа|м2|м3|%|кВт|В|А|°с|°C|т|кг|л|сек|мин|час|сут|м/с|г/л|мг/л)", re.IGNORECASE)

def unit_count_func(text):
    try:
        if not isinstance(text, str):
            return 0
        return len(UNIT_RE.findall(text))
    except:
        return 0

def is_header_safe(text):
    try:
        if text is None or not isinstance(text, str):
            return 0
        t = text.strip()
        if not t:
            return 0
        return int(len(t) < 80 and (t.endswith(':') or re.fullmatch(r"^[A-ZА-Я0-9\.\- ]{3,50}$", t)))
    except:
        return 0

def has_clause_func(text):
    try:
        if not isinstance(text, str):
            return 0
        return int(bool(re.search(r'(?:п\.|пункт)\s*\d+', text, re.IGNORECASE)))
    except:
        return 0

def build_features_for_texts(texts):
    n = len(texts)
    df_all = pd.DataFrame({'text': texts})
    df_all['text_len'] = df_all['text'].apply(lambda t: len(str(t)) if t is not None else 0)
    df_all['norm_refs'] = df_all['text'].apply(lambda t: count_occurrences(t, r'(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+'))
    df_all['imperative'] = df_all['text'].apply(lambda t: count_occurrences(t, r'(должен|должны|следует|требуется|необходимо|обеспечить|не допускается)'))
    df_all['unit_count'] = df_all['text'].apply(unit_count_func)
    df_all['num_count'] = df_all['text'].apply(lambda t: len(re.findall(r'\d', str(t))) if t is not None else 0)
    df_all['num_density'] = df_all['num_count'] / (df_all['text_len'] + 1)
    df_all['is_header'] = df_all['text'].apply(is_header_safe)
    df_all['has_clause'] = df_all['text'].apply(has_clause_func)
    df_all['tech_terms'] = df_all['text'].apply(count_tech_terms_safe)
    df_all['position'] = np.arange(n) / max(1, n-1)
    df_all['after_header'] = df_all['is_header'].shift(1, fill_value=0)
    return df_all[['norm_refs','imperative','unit_count','text_len','num_count','num_density',
                    'is_header','has_clause','tech_terms','position','after_header']]

def add_extra_text_features(texts, feature_df):
    modal_pattern = r'\b(должен|должна|должно|должны|следует|не допускается|запрещается|необходимо|обязательно)\b'
    ntd_ref_pattern = r'(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+'
    unit_number_pattern = r'\d+[\.,]?\d*\s*(?:м|мм|см|км|кг|г|т|с|мин|ч|°С|%|Вт|кВт|МПа|кПа|Па|л|м³|кг/м³)'
    feature_df = feature_df.copy()
    feature_df['num_modal'] = [len(re.findall(modal_pattern, t.lower())) for t in texts]
    feature_df['num_ntd_refs'] = [len(re.findall(ntd_ref_pattern, t, re.IGNORECASE)) for t in texts]
    feature_df['has_unit_number'] = [1 if re.search(unit_number_pattern, t) else 0 for t in texts]
    return feature_df


# 4. Вычисление значимости для Ковыкты
features_base = build_features_for_texts(texts_pd)
features_all = add_extra_text_features(texts_pd, features_base)
features_all = features_all[feature_columns]
X_extra = scaler_extra.transform(features_all.astype(np.float32))
X_enhanced = np.hstack([X_pd, X_extra])
probs_all = base_mlp.predict_proba(X_enhanced)[:, 1]
sig_indices = [i for i, p in enumerate(probs_all) if p >= best_thr]

print(f"Загружена модель (порог = {best_thr:.2f})")
print(f"Получено значимых абзацев: {len(sig_indices)} из {len(texts_pd)}")

Загружена модель (порог = 0.35)
Получено значимых абзацев: 367 из 565


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


# 5. БАЗОВЫЙ BASELINE (ПРОСТОЙ КОСИНУСНЫЙ ПОИСК)

In [ ]:
print("\n" + "="*60)
print("BASELINE (базовый режим с косинусным поиском)")
print("="*60)

SYSTEM_PROMPT = """Вы — эксперт по нормоконтролю проектной документации в нефтегазовой отрасли.
Проверьте соответствие абзаца ПД требованиям НТД, используя предоставленных кандидатов.
Правила:
1) Если **ни один** из кандидатов не содержит требований, ПРЯМО относящихся к данному абзацу ПД, вердикт — "неприменимо".
2) Если кандидат содержит КОНКРЕТНОЕ требование, которому абзац ПД удовлетворяет — "соответствует",
   не удовлетворяет — "не соответствует", частично — "частично соответствует".
3) При любом вердикте, КРОМЕ "неприменимо", обязательно укажите id подходящего кандидата в поле "matched_ntd_id".
4) Ответьте строго JSON без дополнительного текста."""

sim_matrix = cosine_similarity(X_pd, X_ntd)
top_k_indices = np.argsort(-sim_matrix, axis=1)[:, :TOP_K]
top_k_scores = np.take_along_axis(sim_matrix, top_k_indices, axis=1)

results_baseline = []
for pd_idx in range(len(texts_pd)):
    candidates = []
    for k in range(TOP_K):
        ntd_idx = top_k_indices[pd_idx, k]
        candidates.append({
            "id": ids_ntd[ntd_idx],
            "text": texts_ntd[ntd_idx],
            "sim": float(top_k_scores[pd_idx, k])
        })
    user_prompt = f"""Абзац ПД:
{texts_pd[pd_idx]}

Кандидаты НТД:
{chr(10).join([f"Кандидат {i+1} (id: {c['id']}, сходство: {c['sim']:.3f}):\n{c['text']}" for i, c in enumerate(candidates)])}

Ответьте JSON:
{{"verdict": "соответствует|не соответствует|частично соответствует|неприменимо",
 "confidence": 0.0-1.0,
 "matched_ntd_id": "id подходящего кандидата или null",
 "reason": "техобоснование"}}"""
    parsed = None
    for attempt in range(3):
        resp = call_yandexgpt(SYSTEM_PROMPT, user_prompt)
        if resp.startswith("API_ERROR"):
            time.sleep(1)
            continue
        json_match = re.search(r'\{.*\}', resp, re.DOTALL)
        if json_match:
            try:
                data = json.loads(json_match.group())
                if all(k in data for k in ["verdict", "confidence", "matched_ntd_id", "reason"]):
                    parsed = data
                    break
            except: pass
        time.sleep(0.5)
    if not parsed:
        parsed = {"verdict": "ошибка", "confidence": 0.0, "matched_ntd_id": None, "reason": "Не удалось распознать ответ"}
    results_baseline.append({
        "pd_index": pd_idx,
        "pd_text": texts_pd[pd_idx],
        "candidates": candidates,
        "llm_verdict": parsed["verdict"],
        "confidence": parsed["confidence"],
        "matched_ntd_id": parsed["matched_ntd_id"],
        "reason": parsed["reason"]
    })
    if (pd_idx+1) % 50 == 0:
        print(f"Обработано {pd_idx+1}/{len(texts_pd)}")

with open(os.path.join(RESULTS_DIR, "baseline_full.json"), "w", encoding="utf-8") as f:
    json.dump(results_baseline, f, ensure_ascii=False, indent=2)

verdicts = Counter(r["llm_verdict"] for r in results_baseline)
print("\nРаспределение вердиктов Baseline:")
for v, cnt in verdicts.most_common():
    print(f"  {v}: {cnt} ({cnt/len(results_baseline)*100:.1f}%)")



BASELINE (базовый режим с косинусным поиском)
Обработано 50/565
Обработано 100/565
Обработано 150/565
Обработано 200/565
Обработано 250/565
Обработано 300/565
Обработано 350/565
Обработано 400/565
Обработано 450/565
Обработано 500/565
Обработано 550/565

Распределение вердиктов Baseline:
  неприменимо: 493 (87.3%)
  соответствует: 71 (12.6%)
  не соответствует: 1 (0.2%)



# 6. СРАВНЕНИЕ МЕТОДОВ ПОИСКА НТД


In [ ]:
LABELS_XLSX = os.path.join(BASE_DIR, "ПД/итоговые метки_со значимостью.xlsx")

In [ ]:
#  Каскадный метод: BM25 Expanded → BM25
def cascade_search_bm25expanded(pd_idx, top_k=10):
    """
    1. BM25 Expanded (Multi‑Query+RM3) определяет наиболее релевантный документ.
    2. Внутри найденного документа базовый BM25 ранжирует все фрагменты
       по исходному тексту абзаца ПД.
    Возвращает список индексов фрагментов НТД (оригинальные ids_ntd).
    """
    # Шаг 1: получаем кандидатов через BM25 Expanded
    expanded_results = search_improved(pd_idx, top_k=3)
    if not expanded_results:
        return []

    # Берём ID самого релевантного документа (базу)
    best_doc_id = ids_ntd[expanded_results[0]]
    doc_base = best_doc_id.rsplit('::', 1)[0] if '::' in best_doc_id else best_doc_id

    # Собираем все индексы фрагментов этого документа
    doc_indices = clause_to_indices.get(doc_base, [])
    if not doc_indices:
        doc_indices = find_document_indices(doc_base)
    if not doc_indices:
        # fallback: возвращаем результаты BM25 Expanded
        return expanded_results[:top_k]

    # Шаг 2: базовый BM25 внутри документа
    text_query = texts_pd[pd_idx]
    query_tokens = preprocess_fast(text_query)
    if not query_tokens:
        return expanded_results[:top_k]

    # Вычисляем BM25 для фрагментов документа
    scores = np.zeros(len(doc_indices))
    for term in query_tokens:
        if term not in idf:
            continue
        for i, global_idx in enumerate(doc_indices):
            if global_idx in orig_to_clean:
                clean_idx = orig_to_clean[global_idx]
                tf = clean_tokens[clean_idx].count(term)
                scores[i] += idf[term] * (tf * (1.5 + 1)) / (tf + 1.5 * (1 - 0.75 + 0.75 * doc_len[clean_idx] / avgdl))
            else:
                tf = texts_ntd[global_idx].lower().count(term)
                scores[i] += idf[term] * (tf * (1.5 + 1)) / (tf + 1.5 * (1 - 0.75 + 0.75 * avgdl))

    # Если BM25 не дал результатов, используем косинусное сходство
    if scores.max() == 0:
        doc_vecs = X_ntd[doc_indices]
        scores = np.dot(doc_vecs, X_pd[pd_idx])

    sorted_local = np.argsort(scores)[::-1]
    top_doc_fragments = [doc_indices[i] for i in sorted_local[:top_k]]
    return top_doc_fragments

In [ ]:
# ФИНАЛЬНОЕ СРАВНЕНИЕ МЕТОДОВ ПОИСКА НТД (с каскадным методом)
print("\n" + "="*60)
print("СРАВНЕНИЕ МЕТОДОВ ПОИСКА НТД (с Cascade: BM25 Expanded → BM25)")
print("="*60)

# Загрузка разметки
def normalize_spaces(s):
    return re.sub(r'\s+', ' ', str(s)).strip()

text_to_idx = {normalize_spaces(t): i for i, t in enumerate(texts_pd)}
df_labels = pd.read_excel(LABELS_XLSX)
df_labels['pd_index'] = df_labels['text'].apply(lambda t: text_to_idx.get(normalize_spaces(t), -1))
df_labels = df_labels[df_labels['pd_index'] != -1]
df_gt = df_labels[df_labels['НТД'].notna() & (df_labels['НТД'].astype(str).str.strip() != '')]

#  Метрики
def dcg_at_k(relevances, k):
    rel = np.array(relevances[:k], dtype=float)
    if rel.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2))
    return np.sum(rel * discounts)

def ndcg_at_k(pred_indices, rel_set, k):
    rels = [1 if idx in rel_set else 0 for idx in pred_indices[:k]]
    dcg = dcg_at_k(rels, k)
    ideal_rels = sorted([1]*min(len(rel_set), k) + [0]*max(0, k - min(len(rel_set), k)), reverse=True)
    idcg = dcg_at_k(ideal_rels, k)
    return (dcg / idcg) if idcg > 0 else 0.0

def dedup_by_base(pred_indices):
    seen = set()
    out = []
    for idx in pred_indices:
        nid = ids_ntd[idx]
        base = nid.rsplit('::', 1)[0] if '::' in nid else nid
        if base not in seen:
            seen.add(base)
            out.append(idx)
    return out

#  Ground Truth
ground_truth_clause = {}
ground_truth_doc = {}
for _, row in df_gt.iterrows():
    pd_idx = int(row['pd_index'])
    ntd_str = str(row['НТД']).strip()
    ids = [x.strip() for x in re.split(r'[;,]+', ntd_str) if x.strip()]
    relevant_clause = []
    relevant_doc = set()
    for nid in ids:
        if nid in ids_ntd:
            idx = ids_ntd.index(nid)
            relevant_clause.append(idx)
            base = nid.rsplit('::', 1)[0] if '::' in nid else nid
            relevant_doc.update(clause_to_indices.get(base, [idx]))
        else:
            for i, full_id in enumerate(ids_ntd):
                if nid.lower() in full_id.lower() or full_id.lower() in nid.lower():
                    relevant_clause.append(i)
                    base = full_id.rsplit('::', 1)[0] if '::' in full_id else full_id
                    relevant_doc.update(clause_to_indices.get(base, [i]))
                    break
    if relevant_clause:
        ground_truth_clause[pd_idx] = set(relevant_clause)
        ground_truth_doc[pd_idx] = relevant_doc

print(f"Размеченных запросов: {len(ground_truth_clause)}")

#  Раздельная оценка метрик
def calc_metrics_split(pd_idx, pred_indices, rel_clause, rel_doc, ks=[3,5]):
    res = {}
    preds_clause = pred_indices[:]
    preds_doc = dedup_by_base(pred_indices)
    for k in ks:
        top_clause = preds_clause[:k]
        top_doc = preds_doc[:k]
        # Clause-level
        res[f'ClauseHit@{k}'] = int(any(idx in rel_clause for idx in top_clause))
        res[f'ClauseP@{k}'] = (sum(1 for idx in top_clause if idx in rel_clause) / k) if k > 0 else 0.0
        res[f'ClauseMRR@{k}'] = next((1.0/(rank+1) for rank, idx in enumerate(top_clause) if idx in rel_clause), 0.0)
        res[f'ClauseNDCG@{k}'] = ndcg_at_k(top_clause, rel_clause, k)
        # Doc-level
        res[f'DocHit@{k}'] = int(any(idx in rel_doc for idx in top_doc))
        res[f'DocP@{k}'] = (sum(1 for idx in top_doc if idx in rel_doc) / k) if k > 0 else 0.0
        res[f'DocMRR@{k}'] = next((1.0/(rank+1) for rank, idx in enumerate(top_doc) if idx in rel_doc), 0.0)
        res[f'DocNDCG@{k}'] = ndcg_at_k(top_doc, rel_doc, k)
    return res

#  Словарь id -> индекс
id_to_idx = {nid: i for i, nid in enumerate(ids_ntd)}

#  Каскадный метод: BM25 Expanded → BM25
def cascade_search_bm25expanded(pd_idx, top_k=10):
    """
    1. BM25 Expanded (Multi‑Query+RM3) определяет наиболее релевантный документ.
    2. Внутри найденного документа базовый BM25 ранжирует все фрагменты
       по исходному тексту абзаца ПД.
    Возвращает список индексов фрагментов НТД (оригинальные ids_ntd).
    """
    # Шаг 1: получаем кандидатов через BM25 Expanded
    expanded_results = search_improved(pd_idx, top_k=3)
    if not expanded_results:
        return []

    # Берём ID самого релевантного документа (базу)
    best_doc_id = ids_ntd[expanded_results[0]]
    doc_base = best_doc_id.rsplit('::', 1)[0] if '::' in best_doc_id else best_doc_id

    # Собираем все индексы фрагментов этого документа
    doc_indices = clause_to_indices.get(doc_base, [])
    if not doc_indices:
        doc_indices = find_document_indices(doc_base)
    if not doc_indices:
        # fallback: возвращаем результаты BM25 Expanded
        return expanded_results[:top_k]

    # Шаг 2: базовый BM25 внутри документа
    text_query = texts_pd[pd_idx]
    query_tokens = preprocess_fast(text_query)
    if not query_tokens:
        return expanded_results[:top_k]

    # Вычисляем BM25 для фрагментов документа
    scores = np.zeros(len(doc_indices))
    for term in query_tokens:
        if term not in idf:
            continue
        for i, global_idx in enumerate(doc_indices):
            if global_idx in orig_to_clean:
                clean_idx = orig_to_clean[global_idx]
                tf = clean_tokens[clean_idx].count(term)
                scores[i] += idf[term] * (tf * (1.5 + 1)) / (tf + 1.5 * (1 - 0.75 + 0.75 * doc_len[clean_idx] / avgdl))
            else:
                # для неочищенных фрагментов используем частоту в сыром тексте
                tf = texts_ntd[global_idx].lower().count(term)
                scores[i] += idf[term] * (tf * (1.5 + 1)) / (tf + 1.5 * (1 - 0.75 + 0.75 * avgdl))

    # Если BM25 не дал результатов, используем косинусное сходство
    if scores.max() == 0:
        doc_vecs = X_ntd[doc_indices]
        scores = np.dot(doc_vecs, X_pd[pd_idx])

    sorted_local = np.argsort(scores)[::-1]
    top_doc_fragments = [doc_indices[i] for i in sorted_local[:top_k]]
    return top_doc_fragments

#  Методы поиска
methods_ntd = {
    "Regex Direct": lambda pd_idx: regex_candidates(pd_idx, 10),
    "BM25 (all)": lambda pd_idx: (
        (lambda scores: [clean_to_orig[i] for i in np.argsort(scores)[-30:][::-1] if scores[i] > 0])
        (compute_bm25_scores_for_text(texts_pd[pd_idx]))
    )[:10],
    "Universal (theme+fuzzy)": lambda pd_idx: search_universal_enhanced(pd_idx, 10),
    "BM25 Expanded (Multi‑Query+RM3)": lambda pd_idx: search_improved(pd_idx, top_k=10),
    "HyDE (LLM)": lambda pd_idx: hyde_search(pd_idx, top_k=10),
    "Multi‑Channel RRF (MMR)": lambda pd_idx: [
        id_to_idx[item['id']] for item in get_ntd_candidates_final(pd_idx, top_k=10, mmr_lambda=0.85)
    ],
    "Cascade (BM25 Exp. → BM25)": lambda pd_idx: cascade_search_bm25expanded(pd_idx, top_k=10)
}

#  Вычисление метрик
results_ntd = []
for pd_idx in ground_truth_clause:
    for name, func in methods_ntd.items():
        t0 = time.time()
        try:
            preds = func(pd_idx)
        except Exception as e:
            print(f"Ошибка в методе {name} для запроса {pd_idx}: {e}")
            preds = []
        elapsed = (time.time() - t0) * 1000
        if not preds:
            m = {}
            for k in [3,5]:
                m[f'ClauseHit@{k}'] = 0
                m[f'ClauseP@{k}'] = 0.0
                m[f'ClauseMRR@{k}'] = 0.0
                m[f'ClauseNDCG@{k}'] = 0.0
                m[f'DocHit@{k}'] = 0
                m[f'DocP@{k}'] = 0.0
                m[f'DocMRR@{k}'] = 0.0
                m[f'DocNDCG@{k}'] = 0.0
        else:
            m = calc_metrics_split(pd_idx, preds, ground_truth_clause[pd_idx], ground_truth_doc.get(pd_idx, set()))
        m.update({'Method': name, 'Query': pd_idx, 'Time_ms': elapsed})
        results_ntd.append(m)

#  Сводная таблица
df_ntd = pd.DataFrame(results_ntd)
summary_ntd = df_ntd.groupby('Method').agg({
    'ClauseHit@3':'mean','ClauseP@3':'mean','ClauseMRR@3':'mean','ClauseNDCG@3':'mean',
    'DocHit@3':'mean','DocP@3':'mean','DocMRR@3':'mean','DocNDCG@3':'mean',
    'Time_ms':'mean'
}).reset_index()

print("\nРезультаты сравнения (clause — без дедупликации, doc — с дедупликацией):")
print(summary_ntd.to_string(index=False))
summary_ntd.to_csv(os.path.join(RESULTS_DIR, 'ntd_retrieval_final_cascade.csv'), index=False)


СРАВНЕНИЕ МЕТОДОВ ПОИСКА НТД (с Cascade: BM25 Expanded → BM25)
Размеченных запросов: 38

Результаты сравнения (clause — без дедупликации, doc — с дедупликацией):
                         Method  ClauseHit@3  ClauseP@3  ClauseMRR@3  ClauseNDCG@3  DocHit@3   DocP@3  DocMRR@3  DocNDCG@3     Time_ms
                     BM25 (all)     0.421053   0.157895     0.350877      0.286212  0.684211 0.245614  0.570175   0.295355  648.412535
BM25 Expanded (Multi‑Query+RM3)     0.131579   0.043860     0.100877      0.064981  0.921053 0.307018  0.907895   0.427673 1501.952403
     Cascade (BM25 Exp. → BM25)     0.526316   0.210526     0.486842      0.393149  0.894737 0.298246  0.894737   0.419881 1593.800087
                     HyDE (LLM)     0.105263   0.035088     0.074561      0.038665  0.921053 0.307018  0.890351   0.421498 1497.753375
        Multi‑Channel RRF (MMR)     0.131579   0.043860     0.083333      0.053652  0.947368 0.315789  0.811404   0.397092 4132.164058
                   Regex Di

# 7. ПРЯМОЕ ФОРМИРОВАНИЕ КОНТЕКСТА ИЗ КОРПУСА ПД

In [ ]:
print("\n" + "="*60)
print("ПРЯМОЕ ФОРМИРОВАНИЕ КОНТЕКСТА ПД (4 метода)")
print("="*60)

class BM25Wrapper:
    def __init__(self, corpus):
        self.corpus = corpus
        self.tokenized = [tokenize_russian(t) for t in corpus]
        if BM25_AVAILABLE:
            self.bm25 = BM25Okapi(self.tokenized)
        else:
            self.vectorizer = TfidfVectorizer(tokenizer=tokenize_russian, lowercase=False)
            self.tfidf_mat = self.vectorizer.fit_transform(corpus)
    def search(self, query, top_k=20, restrict_indices=None):
        if not query: return []
        if BM25_AVAILABLE:
            scores = self.bm25.get_scores(tokenize_russian(query))
        else:
            q_vec = self.vectorizer.transform([query])
            scores = (self.tfidf_mat @ q_vec.T).toarray().ravel()
        if restrict_indices is not None:
            cand = [(i, scores[i]) for i in restrict_indices]
        else:
            cand = list(enumerate(scores))
        cand.sort(key=lambda x: x[1], reverse=True)
        return cand[:top_k]

class ContextBuilder:
    def __init__(self, texts, X):
        self.texts = texts; self.X = X; self.last_indices = []
    def _local_neighbors(self, idx, window=2):
        n = len(self.texts)
        neigh = list(range(max(0, idx-window), min(n, idx+window+1)))
        if idx in neigh: neigh.remove(idx)
        return neigh
    def _format_block(self, indices, title):
        if not indices: return f"{title}:\n(нет)\n"
        lines = [f"[{i}]: {self.texts[i]}" for i in indices]
        return f"{title}:\n" + "\n".join(lines) + "\n"

class LocalOnlyBuilder(ContextBuilder):
    def build(self, idx):
        local = self._local_neighbors(idx, 2)
        self.last_indices = local
        return self._format_block(local, "ЛОКАЛЬНЫЙ КОНТЕКСТ")

class Baseline2BasketBuilder(ContextBuilder):
    def __init__(self, texts, X, top_k_global=5, cos_thr=0.4):
        super().__init__(texts, X)
        self.top_k = top_k_global; self.cos_thr = cos_thr
    def build(self, idx):
        local = self._local_neighbors(idx, 2)
        q = self.X[idx]
        sims = self.X @ q
        mask = np.ones(len(self.texts), dtype=bool)
        mask[idx] = False; mask[local] = False
        cand_idx = np.argsort(sims[mask])[-self.top_k:][::-1]
        all_idx = np.arange(len(self.texts))[mask][cand_idx]
        global_idx = [int(i) for i in all_idx if sims[i] >= self.cos_thr]
        self.last_indices = local + global_idx
        return (self._format_block(local, "ЛОКАЛЬНЫЙ") + self._format_block(global_idx, "ГЛОБАЛЬНЫЙ СЕМАНТИЧЕСКИЙ"))

class AnchorDrivenBuilder(ContextBuilder):
    def __init__(self, texts, X, bm25_index, top_m=12, top_k_sem=5, cos_thr=0.4):
        super().__init__(texts, X)
        self.bm25 = bm25_index; self.top_m = top_m; self.top_k_sem = top_k_sem; self.cos_thr = cos_thr
    def extract_anchors(self, text):
        nums = re.findall(r"\d+[.,]?\d*\s*(?:мм|см|м|км|МПа|бар|°С|°C|т|кг|л|сек|мин|час|сут)", text, re.IGNORECASE)
        refs = re.findall(r"(?:СП|ГОСТ|СНиП|СанПиН|СТО)\s*[\d\.\-]+", text, re.IGNORECASE)
        verbs = re.findall(r"(должен|должны|следует|требуется|необходимо|обеспечить|предусмотреть)", text, re.IGNORECASE)
        terms = re.findall(r"(трубопровод|задвижка|клапан|насос|свая|фундамент|арматура|кабель|бетон|сталь)", text, re.IGNORECASE)
        return nums + refs + verbs + terms
    def build(self, idx):
        local = self._local_neighbors(idx, 2)
        target = self.texts[idx]
        anchors = self.extract_anchors(target)
        query = " ".join(anchors + [target])
        n = len(self.texts)
        left, right = max(0, idx-10), min(n-1, idx+10)
        restrict = list(range(left, right+1))
        bm25_hits = self.bm25.search(query, top_k=self.top_m, restrict_indices=restrict)
        bm25_cand = [i for i, s in bm25_hits if i != idx and i not in local]
        q_vec = self.X[idx]
        cand_sims = [(i, float(self.X[i] @ q_vec)) for i in bm25_cand]
        cand_sims.sort(key=lambda x: x[1], reverse=True)
        selected = []
        for i, sim in cand_sims:
            if len(selected) >= 4: break
            if not any((self.X[i] @ self.X[j]) >= 0.85 for j in selected):
                selected.append(i)
        cov = compute_anchor_coverage(target, selected, self.texts)
        need_fill = len(selected) < 2 or cov < 0.5
        semantic_sel = []
        if need_fill:
            sims = self.X @ q_vec
            mask = np.ones(len(self.texts), dtype=bool)
            mask[idx] = False; mask[local] = False; mask[selected] = False
            glob_idx = np.argsort(sims[mask])[-self.top_k_sem:][::-1]
            glob_all = np.arange(len(self.texts))[mask][glob_idx]
            semantic_sel = [int(i) for i in glob_all if sims[i] >= self.cos_thr][:max(0, 4-len(selected))]
            selected.extend(semantic_sel)
        self.last_indices = local + selected
        return (self._format_block(local, "ЛОКАЛЬНЫЙ") +
                self._format_block(selected[:len(selected)-len(semantic_sel)], "ЯКОРНЫЙ ПОИСК") +
                self._format_block(semantic_sel, "СЕМАНТИЧЕСКОЕ ДОПОЛНЕНИЕ"))

class StructureAwareBuilder(ContextBuilder):
    def __init__(self, texts, X, blocks, top_k_fallback=3, cos_thr=0.4):
        super().__init__(texts, X)
        self.blocks = blocks; self.top_k_fallback = top_k_fallback; self.cos_thr = cos_thr
    @staticmethod
    def has_discourse_markers(text):
        return bool(re.search(r"(должен|должны|следует|требуется|в соответствии с|п\.\s*\d+|категория|класс|тип)", text, re.IGNORECASE))
    def build(self, idx):
        s, e = find_block_for_index(self.blocks, idx)
        left, right = max(s, idx-3), min(e, idx+3)
        block_cands = [i for i in range(left, right+1) if i != idx]
        disc = [i for i in block_cands if self.has_discourse_markers(self.texts[i])]
        rest = [i for i in block_cands if i not in disc]
        rest_sorted = sorted(rest, key=lambda j: float(self.X[j] @ self.X[idx]), reverse=True)
        struct_sel = (disc + rest_sorted)[:6]
        fallback = []
        if len(struct_sel) < 3 or len(disc) == 0:
            q = self.X[idx]
            sims = self.X @ q
            mask = np.ones(len(self.texts), dtype=bool)
            mask[idx] = False; mask[struct_sel] = False
            glob_idx = np.argsort(sims[mask])[-self.top_k_fallback:][::-1]
            glob_all = np.arange(len(self.texts))[mask][glob_idx]
            fallback = [int(i) for i in glob_all if sims[i] >= self.cos_thr][:max(0, 3-len(struct_sel))]
            struct_sel.extend(fallback)
        self.last_indices = struct_sel
        return (self._format_block(struct_sel[:len(struct_sel)-len(fallback)], "СТРУКТУРНЫЙ БЛОК") +
                self._format_block(fallback, "ДОПОЛНИТЕЛЬНЫЕ"))

def compute_anchor_coverage(target_text, ctx_indices, corpus_texts):
    target_nums = re.findall(r"\d+[.,]?\d*\s*(?:мм|см|м|км|МПа|бар|°С|°C|т|кг|л|сек|мин|час|сут)", target_text, re.IGNORECASE)
    target_refs = re.findall(r"(?:СП|ГОСТ|СНиП|СанПиН|СТО)\s*[\d\.\-]+", target_text, re.IGNORECASE)
    target_verbs = re.findall(r"(должен|должны|следует|требуется|необходимо|обеспечить|предусмотреть)", target_text, re.IGNORECASE)
    target_terms = re.findall(r"(трубопровод|задвижка|клапан|насос|свая|фундамент|арматура|кабель|бетон|сталь)", target_text, re.IGNORECASE)
    total = len(target_nums) + len(target_refs) + len(target_verbs) + len(target_terms)
    if total == 0: return 1.0
    ctx_text = " ".join([corpus_texts[i] for i in ctx_indices]).lower()
    found = 0
    for a in target_refs + target_verbs + target_terms:
        if a.lower() in ctx_text: found += 1
    def parse_num_unit(s):
        m = re.match(r"(\d+[.,]?\d*)\s*(.*)", s)
        if not m: return None
        val = float(m.group(1).replace(',', '.'))
        unit = m.group(2).lower()
        if unit in ('мм',): return (val/1000.0, 'м')
        if unit in ('см',): return (val/100.0, 'м')
        if unit in ('м',): return (val, 'м')
        if unit in ('км',): return (val*1000.0, 'м')
        if unit in ('сек',): return (val, 'с')
        if unit in ('мин',): return (val*60.0, 'с')
        if unit in ('час',): return (val*3600.0, 'с')
        if unit in ('кг',): return (val, 'кг')
        if unit in ('т',): return (val*1000.0, 'кг')
        if unit in ('мпа',): return (val*1e6, 'Па')
        if unit in ('бар',): return (val*1e5, 'Па')
        if unit in ('°с','°c'): return (val, '°C')
        if unit in ('л',): return (val/1000.0, 'м³')
        return None
    target_parsed = [parse_num_unit(s) for s in target_nums]
    target_parsed = [p for p in target_parsed if p is not None]
    ctx_nums = re.findall(r"\d+[.,]?\d*\s*(?:мм|см|м|км|МПа|бар|°С|°C|т|кг|л|сек|мин|час|сут)", ctx_text, re.IGNORECASE)
    ctx_parsed = [parse_num_unit(s) for s in ctx_nums]
    ctx_parsed = [p for p in ctx_parsed if p is not None]
    for (v_t, u_t) in target_parsed:
        for (v_c, u_c) in ctx_parsed:
            if u_t == u_c and v_c > 0 and abs(v_t - v_c) / v_c <= 0.05:
                found += 1
                break
    return found / total


ПРЯМОЕ ФОРМИРОВАНИЕ КОНТЕКСТА ПД (4 метода)


In [ ]:


blocks = build_struct_blocks(texts_pd)
bm25_index = BM25Wrapper(texts_pd)

builders = {
    "Local Only": LocalOnlyBuilder(texts_pd, X_pd),
    "Baseline 2-Basket": Baseline2BasketBuilder(texts_pd, X_pd),
    "Anchor-Driven Hybrid": AnchorDrivenBuilder(texts_pd, X_pd, bm25_index),
    "Structure-Aware + Discourse": StructureAwareBuilder(texts_pd, X_pd, blocks)
}

limit =  len(sig_indices)
use_indices = sig_indices[:limit]
print(f"Оценка методов контекста на {len(use_indices)} абзацах")

RUBRIC = ("Оцените релевантность контекста для проверки целевого абзаца ПД по шкале 1–5:\n"
          "1 – нерелевантен, 3 – частично полезен, 5 – идеально дополняет.\n"
          "Ответьте только цифрой от 1 до 5.")

def evaluate_context(pd_text, context, n_runs=3):
    pd_s = safe_truncate(pd_text, 1500)
    ctx_s = safe_truncate(context, 2500)
    prompt = f"Целевой абзац ПД:\n{pd_s}\n\nКонтекст:\n{ctx_s}\n"
    scores = []
    for _ in range(n_runs):
        resp = call_yandexgpt(RUBRIC, prompt, max_tokens=10)
        m = re.search(r"\b([1-5])\b", resp or "")
        scores.append(int(m.group(1)) if m else 3)
        time.sleep(0.8)
    scores.sort()
    return scores[len(scores)//2]

results_ctx = []
for k, idx in enumerate(use_indices, 1):
    pd_text = texts_pd[idx]
    for name, builder in builders.items():
        t0 = time.time()
        ctx = builder.build(idx)
        t_ms = (time.time() - t0) * 1000
        llm_score = evaluate_context(pd_text, ctx, n_runs=3)
        cov = compute_anchor_coverage(pd_text, builder.last_indices, texts_pd)
        s, e = find_block_for_index(blocks, idx)
        sl = sum(1 for j in builder.last_indices if s <= j <= e) / max(1, len(builder.last_indices))
        if len(builder.last_indices) >= 2:
            vecs = X_pd[builder.last_indices]
            red = np.mean(cosine_similarity(vecs)[np.triu_indices(len(builder.last_indices), k=1)])
        else:
            red = 0.0
        results_ctx.append({
            "idx": idx, "method": name, "llm_score": llm_score,
            "anchor_coverage": cov, "struct_locality": sl, "redundancy": red,
            "context_length": len(ctx), "processing_time_ms": t_ms
        })
    if k % 10 == 0:
        print(f"Обработано {k}/{len(use_indices)}")

df_ctx = pd.DataFrame(results_ctx)
summary_ctx = df_ctx.groupby("method").agg(
    LLM_Score=("llm_score", "mean"),
    Anchor_Coverage=("anchor_coverage", "mean"),
    Structural_Locality=("struct_locality", "mean"),
    Redundancy=("redundancy", "mean"),
    Context_Length=("context_length", "mean"),
    Time_ms=("processing_time_ms", "mean")
).reset_index()

print("\nСравнение методов формирования контекста:")
print(summary_ctx.to_string(index=False))
summary_ctx.to_csv(os.path.join(RESULTS_DIR, "context_pd_comparison.csv"), index=False)

# Сохраняем читаемые результаты (целевой текст, контекст, метрики)
readable_ctx = []
for idx in use_indices:
    target = texts_pd[idx]
    for method_name, builder in builders.items():
        ctx_text = builder.build(idx)
        ctx_indices = builder.last_indices[:]
        ctx_fragments = [{"index": i, "id": ids_pd[i], "text": texts_pd[i]} for i in ctx_indices]
        mask = (df_ctx["idx"] == idx) & (df_ctx["method"] == method_name)
        metrics = {}
        if mask.any():
            row = df_ctx[mask].iloc[0]
            metrics = {"llm_score": int(row["llm_score"]), "anchor_coverage": float(row["anchor_coverage"]),
                       "struct_locality": float(row["struct_locality"]), "redundancy": float(row["redundancy"]),
                       "context_length": int(row["context_length"]), "processing_time_ms": float(row["processing_time_ms"])}
        readable_ctx.append({
            "target_index": int(idx), "target_id": ids_pd[idx], "target_text": target,
            "method": method_name, "context": ctx_text, "context_fragments": ctx_fragments,
            "metrics": metrics
        })

with open(os.path.join(RESULTS_DIR, "context_pd_readable.json"), "w", encoding="utf-8") as f:
    json.dump(readable_ctx, f, ensure_ascii=False, indent=2)

pd.DataFrame(readable_ctx).to_csv(os.path.join(RESULTS_DIR, "context_pd_readable.csv"), index=False)
print("Результаты контекста сохранены.")



ПРЯМОЕ ФОРМИРОВАНИЕ КОНТЕКСТА ПД (4 метода)
Оценка методов контекста на 379 абзацах
Обработано 10/379
Обработано 20/379
Обработано 30/379
Обработано 40/379
Обработано 50/379
Обработано 60/379
Обработано 70/379
Обработано 80/379
Обработано 90/379
Обработано 100/379
Обработано 110/379
Обработано 120/379
Обработано 130/379
Обработано 140/379
Обработано 150/379
Обработано 160/379
Обработано 170/379
Обработано 180/379
Обработано 190/379
Обработано 200/379
Обработано 210/379
Обработано 220/379
Обработано 230/379
Обработано 240/379
Обработано 250/379
Обработано 260/379
Обработано 270/379
Обработано 280/379
Обработано 290/379
Обработано 300/379
Обработано 310/379
Обработано 320/379
Обработано 330/379
Обработано 340/379
Обработано 350/379
Обработано 360/379
Обработано 370/379

Сравнение методов формирования контекста:
                     method  LLM_Score  Anchor_Coverage  Structural_Locality  Redundancy  Context_Length  Time_ms
       Anchor-Driven Hybrid   3.311346         0.638306         

# 8. НОРМАТИВНО-ОРИЕНТИРОВАННОЕ ФОРМИРОВАНИЕ КОНТЕКСТА

In [ ]:
print("\n" + "="*60)
print("НОРМАТИВНО-ОРИЕНТИРОВАННЫЙ КОНТЕКСТ (4 метода, каскадный поиск якоря)")
print("="*60)

#  Вспомогательные функции
def rrf_combine(ranked_lists, k=60):
    scores = {}
    for rlist in ranked_lists:
        for rank, doc_id in enumerate(rlist):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return scores

def mmr_select(candidates, relevance, X, lam=0.6, max_items=10):
    selected = []
    pool = set(candidates)
    while pool and len(selected) < max_items:
        best_id, best_score = None, -1e18
        for cid in pool:
            rel = relevance.get(cid, 0.0)
            sim = max([float(np.dot(X[cid], X[s])) for s in selected]) if selected else 0.0
            score = lam * rel - (1.0 - lam) * sim
            if score > best_score:
                best_score, best_id = score, cid
        if best_id is None:
            break
        selected.append(best_id)
        pool.remove(best_id)
    return selected

def bm25_search_pd(query_text, topk=200):
    """Поиск по корпусу ПД с помощью BM25."""
    hits = bm25_pd.search(query_text, top_k=topk)
    return [i for i, _ in hits]

#  Каскадный поиск якоря НТД (BM25 Expanded → BM25)
def cascade_search_bm25expanded(pd_idx, top_k=10):
    """
    Каскадный метод:
    1. BM25 Expanded (Multi‑Query+RM3) определяет наиболее релевантный документ.
    2. Внутри найденного документа базовый BM25 ранжирует фрагменты по исходному запросу.
    Возвращает список индексов фрагментов НТД (оригинальные ids_ntd).
    """
    # Шаг 1: получаем кандидатов через BM25 Expanded
    expanded_results = search_improved(pd_idx, top_k=3)
    if not expanded_results:
        return []

    # Берём ID самого релевантного документа (базу)
    best_doc_id = ids_ntd[expanded_results[0]]
    doc_base = best_doc_id.rsplit('::', 1)[0] if '::' in best_doc_id else best_doc_id

    # Собираем все индексы фрагментов этого документа
    doc_indices = clause_to_indices.get(doc_base, [])
    if not doc_indices:
        doc_indices = find_document_indices(doc_base)
    if not doc_indices:
        # fallback: возвращаем результаты BM25 Expanded
        return expanded_results[:top_k]

    # Шаг 2: базовый BM25 внутри документа
    text_query = texts_pd[pd_idx]
    query_tokens = preprocess_fast(text_query)
    if not query_tokens:
        return expanded_results[:top_k]

    # Вычисляем BM25 для фрагментов документа
    scores = np.zeros(len(doc_indices))
    for term in query_tokens:
        if term not in idf:
            continue
        for i, global_idx in enumerate(doc_indices):
            if global_idx in orig_to_clean:
                clean_idx = orig_to_clean[global_idx]
                tf = clean_tokens[clean_idx].count(term)
                scores[i] += idf[term] * (tf * (1.5 + 1)) / (tf + 1.5 * (1 - 0.75 + 0.75 * doc_len[clean_idx] / avgdl))
            else:
                # для неочищенных фрагментов используем частоту в сыром тексте
                tf = texts_ntd[global_idx].lower().count(term)
                scores[i] += idf[term] * (tf * (1.5 + 1)) / (tf + 1.5 * (1 - 0.75 + 0.75 * avgdl))

    # Если BM25 не дал результатов, используем косинусное сходство
    if scores.max() == 0:
        doc_vecs = X_ntd[doc_indices]
        scores = np.dot(doc_vecs, X_pd[pd_idx])

    sorted_local = np.argsort(scores)[::-1]
    top_doc_fragments = [doc_indices[i] for i in sorted_local[:top_k]]
    return top_doc_fragments

#  Поиск якоря с каскадным методом
def find_dynamic_anchor_for_target(target_idx):
    """
    Поиск нормативного якоря для абзаца ПД:
    1. Сначала проверяются явные упоминания нормативных документов в тексте абзаца.
    2. Если явных нет, используется каскадный поиск (BM25 Expanded → BM25).
    3. Резервный вариант – косинусное сходство (dense retrieval).
    """
    text = texts_pd[target_idx]

    # 1. Явные упоминания (СП, ГОСТ и т.п.)
    mentions = re.findall(r"\b(СП|ГОСТ|СНиП|СТО|СТУ)\s*[\d\-\.:\*]+", text, re.IGNORECASE)
    for m in mentions:
        m_clean = re.sub(r"\s+", " ", m).strip()
        for i, nid in enumerate(ids_ntd):
            if re.search(re.escape(m_clean), nid, re.IGNORECASE):
                return i, ids_ntd[i]

    # 2. Каскадный поиск (расширенный BM25 → базовый BM25)
    try:
        cascade_results = cascade_search_bm25expanded(target_idx, top_k=1)
        if cascade_results:
            best_idx = cascade_results[0]
            return best_idx, ids_ntd[best_idx]
    except Exception as e:
        # fallback – игнорируем ошибку и переходим к dense
        pass

    # 3. Косинусное сходство (dense retrieval)
    sims = np.dot(X_ntd, X_pd[target_idx])
    top_idx = np.argmax(sims)
    return top_idx, ids_ntd[top_idx]

def get_block_id(idx):
    # Используем идентификатор абзаца ПД для выделения блока
    parts = re.split(r'[:/|#]', ids_pd[idx])
    return parts[0] if parts else "global"

def dedup(indices, X, thr=0.85):
    res = []
    for i in indices:
        if not any(np.dot(X[i], X[j]) > thr for j in res):
            res.append(i)
    return res

#  Базовый класс
class ImprovedNormativeBuilder:
    def __init__(self, max_ctx_chars=2500, max_target_chars=1500, max_anchor_chars=1000):
        self.max_ctx = max_ctx_chars; self.max_tgt = max_target_chars; self.max_anc = max_anchor_chars
        self.last_indices = []; self.anchor_id = None
    def get_anchor(self, idx):
        a_idx, a_id = find_dynamic_anchor_for_target(idx)
        self.anchor_id = a_id
        return a_idx, a_id, texts_ntd[a_idx]
    def pack(self, target_idx, selected, anchor_text, prepend=None):
        parts = []
        if prepend: parts.append(prepend)
        parts.append(f"[Целевой абзац ПД]\n{safe_truncate(texts_pd[target_idx], self.max_tgt)}")
        body = "\n\n".join(f"[ПД {i}]\n{texts_pd[i]}" for i in selected)
        parts.append(body)
        parts.append(f"[Якорная норма НТД]\n{safe_truncate(anchor_text, self.max_anc)}")
        final = ""
        for p in parts:
            if len(final) + len(p) + 2 <= self.max_ctx:
                final = f"{final}\n\n{p}" if final else p
            else:
                rem = self.max_ctx - len(final) - 2
                if rem > 0: final = f"{final}\n\n{safe_truncate(p, rem)}"
                break
        return final
    def metrics(self, target_idx, ctx_text, proc_time):
        indices = self.last_indices
        anchor_text = texts_ntd[self.get_anchor(target_idx)[0]]
        cov_num = len(set(re.findall(r"\d+[.,]?\d*", anchor_text)) & set(re.findall(r"\d+[.,]?\d*", ctx_text))) / max(1, len(set(re.findall(r"\d+[.,]?\d*", anchor_text))))
        tgt_block = get_block_id(target_idx)
        same = sum(1 for i in indices if get_block_id(i) == tgt_block)
        cohesion = same / max(1, len(indices))
        if len(indices) >= 2:
            red = np.mean([np.dot(X_pd[i], X_pd[j]) for i in indices for j in indices if i < j])
        else:
            red = 0.0
        return {
            "DynamicAnchorCoverage": cov_num,
            "SectionCohesion": cohesion,
            "StructuralLocality": same / max(1, len(indices)),
            "Redundancy": red,
            "ContextLength": len(ctx_text),
            "ProcessingTime(ms)": proc_time,
            "NumSegments": len(indices),
            "AnchorId": self.anchor_id or ""
        }

#  Метод A (Dynamic Facet RRF)
class ImprovedA_DynamicFacetRRF(ImprovedNormativeBuilder):
    def _extract_facets(self, anchor_text, target_text):
        nums = re.findall(r"\d+[.,]?\d*\s*(?:мм|см|м|км|МПа|бар|°С|°C|т|кг|л|сек|мин|час|сут)", anchor_text + " " + target_text, re.IGNORECASE)
        mods = ["должен", "должны", "следует", "требуется", "необходимо"]
        mods_present = [m for m in mods if m in (anchor_text + target_text).lower()]
        terms = list(set(re.findall(r"\b[а-яё]{5,}\b", (anchor_text + target_text).lower())))
        refs = re.findall(r"(п\.?\s*\d+(?:\.\d+)*)", anchor_text + target_text, re.IGNORECASE)
        facets = nums + mods_present + terms[:5] + refs
        return list(dict.fromkeys(facets))[:8]
    def build(self, idx):
        a_idx, a_id, a_text = self.get_anchor(idx)
        target = texts_pd[idx]
        facets = self._extract_facets(a_text, target)
        ranked_lists = []
        for f in facets:
            primary = bm25_search_pd(f, topk=140)[:140]
            if primary:
                exp_q = f + " " + " ".join([tokenize_russian(texts_pd[i])[0] for i in primary[:5] if texts_pd[i]])
                secondary = bm25_search_pd(exp_q, topk=120)[:120]
                ranked_lists.append(list(dict.fromkeys(primary + secondary))[:180])
        if not ranked_lists:
            ranked_lists = [bm25_search_pd(target, topk=180)[:180]]
        rrf_scores = rrf_combine(ranked_lists, k=60)
        candidates = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
        filtered = [i for i in candidates if get_block_id(i) == get_block_id(idx) or abs(i - idx) <= 15]
        deduped = dedup(filtered, X_pd)
        rel_adj = {i: rrf_scores[i] * (1.2 if get_block_id(i) == get_block_id(idx) else 0.8) for i in deduped}
        selected = mmr_select(deduped, rel_adj, X_pd, lam=0.6, max_items=12)
        self.last_indices = selected
        return self.pack(idx, selected, a_text)

#  Метод B (Technical Contrast)
TECH_POSITIVE = ["допустимые значения", "нормативные параметры", "соответствует требованиям", "пределы допуска"]
TECH_NEGATIVE = ["аварийный режим", "запрещенные материалы", "предельные отклонения", "несоответствие требованиям"]

class ImprovedB_TechnicalContrast(ImprovedNormativeBuilder):
    def build(self, idx):
        a_idx, a_id, a_text = self.get_anchor(idx)
        target = texts_pd[idx]
        terms = list(set(re.findall(r"\b[а-яё]{5,}\b", (a_text + " " + target).lower())))
        pos_q = " ".join(terms[:5] + TECH_POSITIVE)
        neg_q = " ".join(terms[:5] + TECH_NEGATIVE)
        cand_pos = bm25_search_pd(pos_q, topk=140)[:140]
        cand_neg = bm25_search_pd(neg_q, topk=140)[:140]
        rel = rrf_combine([cand_pos, cand_neg], k=60)
        candidates = sorted(rel, key=rel.get, reverse=True)
        filtered = [i for i in candidates if get_block_id(i) == get_block_id(idx) or abs(i - idx) <= 15]
        deduped = dedup(filtered, X_pd)
        rel_adj = {i: rel[i] * (1.2 if get_block_id(i) == get_block_id(idx) else 0.8) for i in deduped}
        selected = mmr_select(deduped, rel_adj, X_pd, lam=0.6, max_items=12)
        self.last_indices = selected
        ctx_text = "\n".join(texts_pd[i] for i in selected)
        consistency = f"Technical Parameter Consistency: {len(set(re.findall(r'\d+[.,]?\d*', target)) & set(re.findall(r'\d+[.,]?\d*', ctx_text))) / max(1, len(set(re.findall(r'\d+[.,]?\d*', target)))):.2f}"
        return self.pack(idx, selected, a_text, prepend=consistency)

#  Метод C (Coverage Iterative)
class ImprovedC_CoverageIterative(ImprovedNormativeBuilder):
    def build(self, idx):
        a_idx, a_id, a_text = self.get_anchor(idx)
        target = texts_pd[idx]
        pool = bm25_search_pd(target, topk=160)[:160]
        best_ctx = "\n".join(texts_pd[i] for i in pool[:8])
        cov, missing = self._coverage(target, best_ctx)
        prev_cov = 0.0
        rounds = 1
        while rounds < 3 and cov < 0.7 and (cov - prev_cov) >= 0.05:
            prev_cov = cov
            q = " ".join(missing[:8]) if missing else ""
            if q:
                new = bm25_search_pd(q, topk=120)[:120]
                pool = list(dict.fromkeys(pool + new))
                best_ctx = "\n".join(texts_pd[i] for i in pool[:8])
                cov, missing = self._coverage(target, best_ctx)
            rounds += 1
        blk = get_block_id(idx)
        same_blk = [i for i in pool if get_block_id(i) == blk]
        other_blk = [i for i in pool if get_block_id(i) != blk]
        cap_other = int(0.3 * 12)
        constrained = same_blk + other_blk[:cap_other]
        rel = rrf_combine([pool, constrained])
        ordered = sorted(constrained, key=lambda i: rel.get(i, 0.0), reverse=True)
        ordered = dedup(ordered, X_pd)
        selected = mmr_select(ordered, {i: rel.get(i, 0.0) for i in ordered}, X_pd, lam=0.65, max_items=12)
        self.last_indices = selected
        return self.pack(idx, selected, a_text, prepend=f"Coverage: {cov:.2f}")
    def _coverage(self, target, ctx):
        cov_num = len(set(re.findall(r"\d+[.,]?\d*", target)) & set(re.findall(r"\d+[.,]?\d*", ctx))) / max(1, len(set(re.findall(r"\d+[.,]?\d*", target))))
        terms = list(set(tokenize_russian(target)).difference({"и", "в", "на", "с", "по", "для", "не"}))
        terms = [t for t in terms if len(t) > 3]
        ctx_tokens = set(tokenize_russian(ctx))
        missing = [t for t in terms if t not in ctx_tokens]
        cov_term = (len(terms) - len(missing)) / max(1, len(terms))
        score = 0.6 * cov_num + 0.4 * cov_term
        return score, missing

#  Метод D (Citation Local)
class ImprovedD_CitationLocal(ImprovedNormativeBuilder):
    def build(self, idx):
        a_idx, a_id, a_text = self.get_anchor(idx)
        norm_num = re.search(r"(ГОСТ|СП|СНиП|СТО|СТУ)\s*[\d\-\.:\*]+", a_id)
        if norm_num:
            pattern = re.escape(norm_num.group(0))
            cited = [i for i, t in enumerate(texts_pd) if re.search(pattern, t, re.IGNORECASE)]
        else:
            cited = []
        neighborhood = set()
        for c in cited:
            blk = get_block_id(c)
            for j in range(max(0, c - 3), min(len(texts_pd), c + 4)):
                if get_block_id(j) == blk:
                    neighborhood.add(j)
        if len(cited) < 3:
            bm25_res = bm25_search_pd(texts_pd[idx], topk=5)
            neighborhood.update(bm25_res[:5])
        candidates = sorted(neighborhood)
        scores = {}
        for i in candidates:
            cit_strength = 2.0 if i in cited else 0.0
            sem = float(np.dot(X_pd[i], X_pd[idx]))
            scores[i] = 0.6 * cit_strength + 0.4 * sem
        ordered = sorted(candidates, key=lambda i: scores[i], reverse=True)
        deduped = dedup(ordered, X_pd, thr=0.88)
        rel_adj = {i: scores[i] * (1.2 if get_block_id(i) == get_block_id(idx) else 0.8) for i in deduped}
        selected = mmr_select(deduped, rel_adj, X_pd, lam=0.65, max_items=12)
        self.last_indices = selected
        return self.pack(idx, selected, a_text)

#  Индексы BM25 для ПД
if "bm25_pd" not in globals():
    print("Создание BM25-индекса для ПД...")
    bm25_pd = BM25Wrapper(texts_pd)
if "bm25_ntd" not in globals():
    print("Создание BM25-индекса для НТД...")
    bm25_ntd = BM25Wrapper(texts_ntd)

#  Сравнение на значимых абзацах
limit_comp = len(sig_indices)
comp_idx = sig_indices[:limit_comp]

methods_norm = {
    "A_DynamicFacetRRF": ImprovedA_DynamicFacetRRF(),
    "B_TechnicalContrast": ImprovedB_TechnicalContrast(),
    "C_CoverageIter": ImprovedC_CoverageIterative(),
    "D_CitationLocal": ImprovedD_CitationLocal()
}

#  Цикл с отображением прогресса
try:
    from tqdm import tqdm
    pbar = tqdm(comp_idx, desc="Обработка абзацев ПД")
except ImportError:
    pbar = comp_idx
    print("tqdm не установлен, будет выводиться номер каждые 10 абзацев")

results_norm = []
for i, idx in enumerate(pbar if isinstance(pbar, tqdm) else comp_idx):
    if not isinstance(pbar, tqdm) and i % 10 == 0:
        print(f"Обработка абзаца {i+1}/{len(comp_idx)}...")
    for name, builder in methods_norm.items():
        t0 = time.time()
        ctx = builder.build(idx)
        t1 = time.time()
        m = builder.metrics(idx, ctx, (t1 - t0)*1000)
        m["Method"] = name
        m["Index"] = idx
        results_norm.append(m)

df_norm = pd.DataFrame(results_norm)
summary_norm = df_norm.groupby("Method").agg(
    DynamicAnchorCoverage=("DynamicAnchorCoverage", "mean"),
    SectionCohesion=("SectionCohesion", "mean"),
    Redundancy=("Redundancy", "mean"),
    ContextLength=("ContextLength", "mean"),
    ProcessingTime_ms=("ProcessingTime(ms)", "mean")
).reset_index()

print("\nСравнение нормативно-ориентированных методов (каскадный поиск якоря):")
print(summary_norm.to_string(index=False))
summary_norm.to_csv(os.path.join(RESULTS_DIR, "normative_context_comparison.csv"), index=False)

readable_norm = []
for idx in comp_idx:
    target = texts_pd[idx]
    for name, builder in methods_norm.items():
        ctx_text = builder.build(idx)
        readable_norm.append({
            "target_index": idx,
            "method": name,
            "context": ctx_text
        })
pd.DataFrame(readable_norm).to_csv(os.path.join(RESULTS_DIR, "normative_context_readable.csv"), index=False)
print("Результаты нормативно-ориентированного контекста сохранены.")


НОРМАТИВНО-ОРИЕНТИРОВАННЫЙ КОНТЕКСТ (4 метода, каскадный поиск якоря)


Обработка абзацев ПД: 100%|██████████| 379/379 [1:39:53<00:00, 15.81s/it]



Сравнение нормативно-ориентированных методов (каскадный поиск якоря):
             Method  DynamicAnchorCoverage  SectionCohesion  Redundancy  ContextLength  ProcessingTime_ms
  A_DynamicFacetRRF               0.137238         0.089543    0.272095    2470.730871        2052.280863
B_TechnicalContrast               0.264496         0.189062    0.292549    2210.145119        1996.712965
     C_CoverageIter               0.518208         0.264292    0.336155    1866.562005        1940.115057
    D_CitationLocal               0.443800         0.204266    0.332634    2098.833773        1971.910462
Результаты нормативно-ориентированного контекста сохранены.


# 9. ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ: СРАВНЕНИЕ ТРЁХ РЕЖИМОВ

In [ ]:
def get_block_id(idx):
    parts = re.split(r'[:/|#]', ids_pd[idx])
    return parts[0] if parts else "global"

In [ ]:
# Вспомогательная функция для вызова LLM
def get_llm_verdict_soft(pd_text, candidates, context=None):
    """
    Отправляет запрос к YandexGPT, возвращает словарь с ключами:
    verdict, confidence, matched_ntd_id, reason
    """
    system_prompt = """Вы — эксперт по нормоконтролю проектной документации в нефтегазовой отрасли.
Проверьте соответствие абзаца ПД требованиям НТД, используя предоставленных кандидатов.
Правила:
1) Если **ни один** из кандидатов не содержит требований, ПРЯМО относящихся к данному абзацу ПД, вердикт — "неприменимо".
2) Если кандидат содержит КОНКРЕТНОЕ требование, которому абзац ПД удовлетворяет — "соответствует",
   не удовлетворяет — "не соответствует", частично — "частично соответствует".
3) При любом вердикте, КРОМЕ "неприменимо", обязательно укажите id подходящего кандидата в поле "matched_ntd_id".
4) Ответьте строго JSON без дополнительного текста."""

    cand_text = "\n".join(
        f"Кандидат {i+1} (id: {c['id']}, сходство: {c['sim']:.3f}):\n{c['text']}"
        for i, c in enumerate(candidates)
    )
    user_prompt = f"Абзац ПД:\n{pd_text}\n\nКандидаты НТД:\n{cand_text}\n"
    if context:
        user_prompt += f"\nДополнительный контекст ПД:\n{context}\n"

    user_prompt += '\nОтветьте JSON:\n{"verdict": "соответствует|не соответствует|частично соответствует|неприменимо", "confidence": 0.0-1.0, "matched_ntd_id": "id подходящего кандидата или null", "reason": "техобоснование"}'

    parsed = None
    for attempt in range(3):
        resp = call_yandexgpt(system_prompt, user_prompt, max_tokens=500)
        if resp.startswith("API_ERROR"):
            time.sleep(1)
            continue
        json_match = re.search(r'\{.*\}', resp, re.DOTALL)
        if json_match:
            try:
                data = json.loads(json_match.group())
                if all(k in data for k in ["verdict", "confidence", "matched_ntd_id", "reason"]):
                    parsed = data
                    break
            except:
                pass
        time.sleep(0.5)
    if not parsed:
        parsed = {"verdict": "ошибка", "confidence": 0.0, "matched_ntd_id": None, "reason": "Не удалось распознать ответ"}
    return parsed


# ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ: BASELINE⁺ vs STRUCTURE‑AWARE vs NORMATIVE‑AWARE
# (кандидаты НТД – Cascade BM25 Expanded → BM25)

print("\n" + "="*60)
print("ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ: BASELINE⁺ vs STRUCTURE‑AWARE vs NORMATIVE‑AWARE")
print("(кандидаты НТД – Cascade BM25 Expanded → BM25)")
print("="*60)


# Поиск кандидатов НТД – каскадный метод

def get_ntd_candidates_cascade(pd_idx, top_k=8):
    indices = cascade_search_bm25expanded(pd_idx, top_k=top_k)
    result = []
    for idx in indices[:top_k]:
        result.append({
            "id": ids_ntd[idx],
            "text": texts_ntd[idx],
            "sim": float(np.dot(X_ntd[idx], X_pd[pd_idx]))
        })
    return result


# Контекстные билдеры (лучшие методы)
if 'builder_struct' not in dir() or builder_struct is None:
    blocks = build_struct_blocks(texts_pd)
    builder_struct = StructureAwareBuilder(texts_pd, X_pd, blocks)
if 'builder_norm' not in dir() or builder_norm is None:
    builder_norm = ImprovedC_CoverageIterative()

if 'sig_indices' not in dir() or len(sig_indices) == 0:
    sig_indices = list(range(len(texts_pd)))

print(f"Запуск на {len(sig_indices)} абзацах")
results_final = []
start_time = time.time()

for idx_count, pd_idx in enumerate(sig_indices, 1):
    pd_text = texts_pd[pd_idx]
    candidates = get_ntd_candidates_cascade(pd_idx, top_k=8)

    if not candidates:
        results_final.append({
            "pd_index": int(pd_idx), "pd_text": pd_text,
            "baseline_verdict": "неприменимо", "baseline_confidence": 0.0, "baseline_time": 0.0,
            "baseline_reason": "Нет кандидатов",
            "struct_verdict": "неприменимо", "struct_confidence": 0.0, "struct_time": 0.0,
            "struct_reason": "Нет кандидатов",
            "norm_verdict": "неприменимо", "norm_confidence": 0.0, "norm_time": 0.0,
            "norm_reason": "Нет кандидатов",
            "cascade_verdict": "неприменимо", "cascade_confidence": 0.0, "cascade_reason": "Нет кандидатов",
            "cascade_triggered": True
        })
        continue

    # Baseline⁺ (3 кандидата, без контекста)
    t0 = time.time()
    v_base = get_llm_verdict_soft(pd_text, candidates[:3])
    t_base = time.time() - t0

    # Structure‑Aware (8 кандидатов + прямой контекст)
    ctx_struct = builder_struct.build(pd_idx)
    t0 = time.time()
    v_struct = get_llm_verdict_soft(pd_text, candidates, context=ctx_struct)
    t_struct = time.time() - t0

    # Normative‑Aware (8 кандидатов + нормативный контекст)
    ctx_norm = builder_norm.build(pd_idx)
    t0 = time.time()
    v_norm = get_llm_verdict_soft(pd_text, candidates, context=ctx_norm)
    t_norm = time.time() - t0

    # Каскад
    cascade_triggered = False
    if v_struct["confidence"] < 0.5:
        has_good = any(
            section_boost(c['id']) > 1.0 and
            any(s['unit'] in c['text'].lower() for s in extract_constraints_slots(pd_text))
            for c in candidates
        )
        if has_good and candidates:
            v_cascade = {
                "verdict": "частично соответствует",
                "confidence": 0.6,
                "matched_ntd_id": candidates[0]['id'],
                "reason": "Обнаружено тематическое совпадение"
            }
            cascade_triggered = True
        else:
            v_cascade = {
                "verdict": "неприменимо",
                "confidence": 0.0,
                "matched_ntd_id": None,
                "reason": "Низкая уверенность"
            }
    else:
        v_cascade = v_struct

    results_final.append({
        "pd_index": int(pd_idx), "pd_text": pd_text,
        "baseline_verdict": v_base["verdict"], "baseline_confidence": v_base["confidence"],
        "baseline_time": round(t_base, 3), "baseline_reason": v_base.get("reason", ""),
        "struct_verdict": v_struct["verdict"], "struct_confidence": v_struct["confidence"],
        "struct_time": round(t_struct, 3), "struct_reason": v_struct.get("reason", ""),
        "norm_verdict": v_norm["verdict"], "norm_confidence": v_norm["confidence"],
        "norm_time": round(t_norm, 3), "norm_reason": v_norm.get("reason", ""),
        "cascade_verdict": v_cascade["verdict"], "cascade_confidence": v_cascade["confidence"],
        "cascade_reason": v_cascade.get("reason", ""),
        "cascade_triggered": cascade_triggered
    })

    if idx_count % 20 == 0:
        elapsed = time.time() - start_time
        print(f"Обработано {idx_count}/{len(sig_indices)} ({elapsed:.0f} сек)")


# Метрики

df_final = pd.DataFrame(results_final)
summary_final = pd.DataFrame({
    "Режим": ["Baseline⁺", "Structure‑Aware", "Normative‑Aware", "Каскад"],
    "Средняя уверенность": [
        df_final["baseline_confidence"].mean(),
        df_final["struct_confidence"].mean(),
        df_final["norm_confidence"].mean(),
        df_final["cascade_confidence"].mean()
    ],
    "Доля 'неприменимо'": [
        (df_final["baseline_verdict"] == "неприменимо").mean(),
        (df_final["struct_verdict"] == "неприменимо").mean(),
        (df_final["norm_verdict"] == "неприменимо").mean(),
        (df_final["cascade_verdict"] == "неприменимо").mean()
    ],
    "Среднее время (с)": [
        df_final["baseline_time"].mean(),
        df_final["struct_time"].mean(),
        df_final["norm_time"].mean(),
        df_final["struct_time"].mean()
    ]
})

print("\n=== ИТОГОВАЯ ТАБЛИЦА (Cascade BM25 Expanded → BM25) ===")
print(summary_final.to_string(index=False))


# Сохранение основных результатов
out_json = os.path.join(RESULTS_DIR, "final_experiment_cascade.json")
out_csv = os.path.join(RESULTS_DIR, "final_experiment_cascade.csv")
with open(out_json, "w", encoding="utf-8") as f:
    json.dump(results_final, f, ensure_ascii=False, indent=2)
df_final.to_csv(out_csv, index=False)
print(f"Основные результаты сохранены в {out_json}")


# ЧИТАЕМЫЙ ФАЙЛ
print("\nФормирование читаемых результатов...")
readable = []
for idx in sig_indices:
    row = df_final[df_final["pd_index"] == idx].iloc[0]
    cands = get_ntd_candidates_cascade(idx, top_k=5)
    ctx_struct = builder_struct.build(idx)
    ctx_norm = builder_norm.build(idx)

    readable.append({
        "pd_index": int(idx),
        "target_pd_text": texts_pd[idx],
        "ntd_candidates": [{"id": c["id"], "text": c["text"]} for c in cands],
        "context_struct": ctx_struct,
        "context_norm": ctx_norm,
        "baseline_verdict": row["baseline_verdict"],
        "baseline_confidence": row["baseline_confidence"],
        "baseline_reason": row["baseline_reason"],
        "struct_verdict": row["struct_verdict"],
        "struct_confidence": row["struct_confidence"],
        "struct_reason": row["struct_reason"],
        "norm_verdict": row["norm_verdict"],
        "norm_confidence": row["norm_confidence"],
        "norm_reason": row["norm_reason"],
        "cascade_verdict": row["cascade_verdict"],
        "cascade_confidence": row["cascade_confidence"],
        "cascade_reason": row["cascade_reason"]
    })

readable_json = os.path.join(RESULTS_DIR, "readable_final_cascade.json")
with open(readable_json, "w", encoding="utf-8") as f:
    json.dump(readable, f, ensure_ascii=False, indent=2)

pd.DataFrame(readable).to_csv(
    os.path.join(RESULTS_DIR, "readable_final_cascade.csv"),
    index=False
)

print(f"Читаемые результаты сохранены:\n  {readable_json}\n  readable_final_cascade.csv")


ФИНАЛЬНЫЙ ЭКСПЕРИМЕНТ: BASELINE⁺ vs STRUCTURE‑AWARE vs NORMATIVE‑AWARE
(кандидаты НТД – Cascade BM25 Expanded → BM25)
Запуск на 367 абзацах
Обработано 20/367 (293 сек)
Обработано 40/367 (556 сек)
Обработано 60/367 (764 сек)
Обработано 80/367 (974 сек)
Обработано 100/367 (1200 сек)
Обработано 120/367 (1439 сек)
Обработано 140/367 (1666 сек)
Обработано 160/367 (1896 сек)
Обработано 180/367 (2128 сек)
Обработано 200/367 (2351 сек)
Обработано 220/367 (2571 сек)
Обработано 240/367 (2791 сек)
Обработано 260/367 (3025 сек)
Обработано 280/367 (3233 сек)
Обработано 300/367 (3459 сек)
Обработано 320/367 (3679 сек)
Обработано 340/367 (3926 сек)
Обработано 360/367 (4140 сек)

=== ИТОГОВАЯ ТАБЛИЦА (Cascade BM25 Expanded → BM25) ===
          Режим  Средняя уверенность  Доля 'неприменимо'  Среднее время (с)
      Baseline⁺             0.384589            0.365123           2.723910
Structure‑Aware             0.532826            0.297003           2.963477
Normative‑Aware             0.671485      

In [ ]:
TECH_KEYWORDS = [
    "грунт","глубин","свай","бетон","фундамент","отметк","уклон","сталь","кабель",
    "трубопровод","арматур","клапан","давлен","температур","мощность","расстояни",
    "пожарн","защит","сигнализаци","объект","сооружени","здани"
]

def count_tech_terms_safe(text):
    try:
        t = text.lower()
        return sum(1 for kw in TECH_KEYWORDS if kw in t)
    except:
        return 0

def count_occurrences(text, pattern):
    try:
        if not isinstance(text, str): return 0
        return len(re.findall(pattern, text, re.IGNORECASE))
    except: return 0

UNIT_RE = re.compile(r"\d+[.,]?\d*\s*(?:м|мм|см|км|мпа|м2|м3|%|кВт|В|А|°с|°C|т|кг|л|сек|мин|час|сут|м/с|г/л|мг/л)", re.IGNORECASE)

def unit_count_func(text):
    try:
        if not isinstance(text, str): return 0
        return len(UNIT_RE.findall(text))
    except: return 0

def is_header_safe(text):
    try:
        if text is None or not isinstance(text, str): return 0
        t = text.strip()
        if not t: return 0
        return int(len(t) < 80 and (t.endswith(':') or re.fullmatch(r"^[A-ZА-Я0-9\.\- ]{3,50}$", t)))
    except: return 0

def has_clause_func(text):
    try:
        if not isinstance(text, str): return 0
        return int(bool(re.search(r'(?:п\.|пункт)\s*\d+', text, re.IGNORECASE)))
    except: return 0

def build_features_for_texts(texts):
    n = len(texts)
    df_all = pd.DataFrame({'text': texts})
    df_all['text_len'] = df_all['text'].apply(lambda t: len(str(t)) if t is not None else 0)
    df_all['norm_refs'] = df_all['text'].apply(lambda t: count_occurrences(t, r'(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+'))
    df_all['imperative'] = df_all['text'].apply(lambda t: count_occurrences(t, r'(должен|должны|следует|требуется|необходимо|обеспечить|не допускается)'))
    df_all['unit_count'] = df_all['text'].apply(unit_count_func)
    df_all['num_count'] = df_all['text'].apply(lambda t: len(re.findall(r'\d', str(t))) if t is not None else 0)
    df_all['num_density'] = df_all['num_count'] / (df_all['text_len'] + 1)
    df_all['is_header'] = df_all['text'].apply(is_header_safe)
    df_all['has_clause'] = df_all['text'].apply(has_clause_func)
    df_all['tech_terms'] = df_all['text'].apply(count_tech_terms_safe)
    df_all['position'] = np.arange(n) / max(1, n-1)
    df_all['after_header'] = df_all['is_header'].shift(1, fill_value=0)
    return df_all[['norm_refs','imperative','unit_count','text_len','num_count','num_density',
                    'is_header','has_clause','tech_terms','position','after_header']]

def add_extra_text_features(texts, feature_df):
    modal_pattern = r'\b(должен|должна|должно|должны|следует|не допускается|запрещается|необходимо|обязательно)\b'
    ntd_ref_pattern = r'(?:СП|ГОСТ|СНиП|СанПиН|СТО|СТУ)\s*[\d\.\-]+'
    unit_number_pattern = r'\d+[\.,]?\d*\s*(?:м|мм|см|км|кг|г|т|с|мин|ч|°С|%|Вт|кВт|МПа|кПа|Па|л|м³|кг/м³)'
    feature_df = feature_df.copy()
    feature_df['num_modal'] = [len(re.findall(modal_pattern, t.lower())) for t in texts]
    feature_df['num_ntd_refs'] = [len(re.findall(ntd_ref_pattern, t, re.IGNORECASE)) for t in texts]
    feature_df['has_unit_number'] = [1 if re.search(unit_number_pattern, t) else 0 for t in texts]
    return feature_df

def load_pkl_embeddings(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    ids = [str(item["id"]) for item in data]
    texts = [item["text"] for item in data]
    X = np.array([item["embedding"] for item in data], dtype=np.float32)
    return ids, texts, X

In [ ]:
# 1. Загрузка модели и параметров
MODEL_DIR = os.path.join(BASE_DIR, "models")   # папка с сохранённой моделью
base_mlp = joblib.load(os.path.join(MODEL_DIR, "base_mlp.pkl"))
scaler_extra = joblib.load(os.path.join(MODEL_DIR, "scaler_extra.pkl"))
with open(os.path.join(MODEL_DIR, "inference_params.json"), "r", encoding="utf-8") as f:
    params = json.load(f)
best_thr = params["best_threshold"]
feature_columns = params["feature_columns"]


# 2. Загрузка проекта ПД

NEW_PROJECT_PKL = os.path.join(BASE_DIR, "ПД/эмбеддинги/2. ОБУСТРОЙСТВО ЧАЯНДИНСКОГО НГКМ.pkl")  # пример

new_ids, new_texts, new_X = load_pkl_embeddings(NEW_PROJECT_PKL)
new_X = new_X / (np.linalg.norm(new_X, axis=1, keepdims=True) + 1e-12)   # L2


# 3. Построение признаков и масштабирование
features_base = build_features_for_texts(new_texts)
features_all = add_extra_text_features(new_texts, features_base)
features_all = features_all[feature_columns]
X_extra = scaler_extra.transform(features_all.astype(np.float32))
X_enhanced = np.hstack([new_X, X_extra])
probs_all = base_mlp.predict_proba(X_enhanced)[:, 1]
sig_indices_new = [i for i, p in enumerate(probs_all) if p >= best_thr]

print(f"Новый проект: всего абзацев {len(new_texts)}, значимых {len(sig_indices_new)}")


# 4. Временно подменяем глобальные переменные ПД и пересоздаём BM25
old_texts_pd, old_X_pd, old_ids_pd = texts_pd, X_pd, ids_pd
old_bm25_pd = bm25_pd

# Подменяем на новый проект
texts_pd = new_texts
X_pd = new_X
ids_pd = new_ids

# Пересоздаём BM25-индекс для поиска контекста внутри ПД
bm25_pd = BM25Wrapper(texts_pd)

# Создаём нормативный билдер (метод CoverageIter)
builder_norm = ImprovedC_CoverageIterative()


# 5. Сравнение Baseline⁺ и Normative‑Aware
results_compare = []
start_time = time.time()

for cnt, idx in enumerate(sig_indices_new, 1):
    pd_text = texts_pd[idx]
    candidates = get_ntd_candidates_cascade(idx, top_k=8)

    if not candidates:
        results_compare.append({
            "pd_index": int(idx),
            "baseline_verdict": "неприменимо", "baseline_confidence": 0.0, "baseline_time": 0.0,
            "baseline_reason": "Нет кандидатов",
            "norm_verdict": "неприменимо", "norm_confidence": 0.0, "norm_time": 0.0,
            "norm_reason": "Нет кандидатов"
        })
        continue

    # Baseline⁺ (3 кандидата, без контекста)
    t0 = time.time()
    v_base = get_llm_verdict_soft(pd_text, candidates[:3])
    t_base = time.time() - t0

    # Normative‑Aware (8 кандидатов + нормативный контекст)
    ctx_norm = builder_norm.build(idx)
    t0 = time.time()
    v_norm = get_llm_verdict_soft(pd_text, candidates, context=ctx_norm)
    t_norm = time.time() - t0

    results_compare.append({
        "pd_index": int(idx),
        "baseline_verdict": v_base["verdict"], "baseline_confidence": v_base["confidence"],
        "baseline_time": round(t_base, 3), "baseline_reason": v_base.get("reason", ""),
        "norm_verdict": v_norm["verdict"], "norm_confidence": v_norm["confidence"],
        "norm_time": round(t_norm, 3), "norm_reason": v_norm.get("reason", "")
    })

    if cnt % 10 == 0:
        print(f"Обработано {cnt}/{len(sig_indices_new)} ({time.time() - start_time:.0f} сек)")


# 6. Сводка и сохранение
df_comp = pd.DataFrame(results_compare)
summary = pd.DataFrame({
    "Режим": ["Baseline⁺", "Normative‑Aware"],
    "Средняя уверенность": [df_comp["baseline_confidence"].mean(), df_comp["norm_confidence"].mean()],
    "Доля 'неприменимо'": [(df_comp["baseline_verdict"] == "неприменимо").mean(),
                            (df_comp["norm_verdict"] == "неприменимо").mean()],
    "Среднее время (с)": [df_comp["baseline_time"].mean(), df_comp["norm_time"].mean()]
})

print("\n=== РЕЗУЛЬТАТЫ НА НОВОМ ПРОЕКТЕ ===")
print(summary.to_string(index=False))


OUT_DIR = os.path.join(BASE_DIR, "Результаты", "Новый_проект")
os.makedirs(OUT_DIR, exist_ok=True)
df_comp.to_csv(os.path.join(OUT_DIR, "comparison_new_project.csv"), index=False)
with open(os.path.join(OUT_DIR, "comparison_new_project.json"), "w", encoding="utf-8") as f:
    json.dump(results_compare, f, ensure_ascii=False, indent=2)
print(f"Детальные результаты сохранены в {OUT_DIR}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Новый проект: всего абзацев 611, значимых 260
Обработано 10/260 (107 сек)
Обработано 20/260 (208 сек)
Обработано 30/260 (277 сек)
Обработано 40/260 (362 сек)
Обработано 50/260 (442 сек)
Обработано 60/260 (518 сек)
Обработано 70/260 (603 сек)
Обработано 80/260 (684 сек)
Обработано 90/260 (772 сек)
Обработано 100/260 (864 сек)
Обработано 110/260 (947 сек)
Обработано 120/260 (1031 сек)
Обработано 130/260 (1118 сек)
Обработано 140/260 (1205 сек)
Обработано 150/260 (1306 сек)
Обработано 160/260 (1391 сек)
Обработано 170/260 (1466 сек)
Обработано 180/260 (1560 сек)
Обработано 190/260 (1652 сек)
Обработано 200/260 (1734 сек)
Обработано 210/260 (1819 сек)
Обработано 220/260 (1892 сек)
Обработано 230/260 (1980 сек)
Обработано 240/260 (2063 сек)
Обработано 250/260 (2143 сек)
Обработано 260/260 (2224 сек)

=== РЕЗУЛЬТАТЫ НА НОВОМ ПРОЕКТЕ ===
          Режим  Средняя уверенность  Доля 'неприменимо'  Среднее время (с)
      Baseline⁺             0.302619            0.450000           2.724323
Norma